# OREGON 2-YEAR UPLIFT MODEL (T-LEARNER WITH XGBOOST)

Input: `DataSets/Oregon_2YearDataset.csv`  
Binary outcome: `outcome_ed_90d` (1 = one or more ED visits; missing = 0)  
Treatment: `Engaged_flag` (1 = treated/engaged, 0 = control/opted out), validated against `OptOut_flag`

This notebook preserves the Funds Combined notebook's modeling, hyperparameter
search, diagnostics, calibration, Brier scoring, charts, and output methods.
Oregon-specific changes are limited to schema-aware cleaning, treatment derivation,
post-treatment leakage controls, member-grouped splitting, train-fitted preprocessing,
and isolated Oregon outputs.

XGBoost is configured to require the SageMaker CUDA GPU on device 0, matching
the authoritative Funds notebook. Run this notebook on a GPU-backed instance.

The Funds source notebook does not define separate Qini or cumulative-gain
calculations. Its cumulative uplift-by-targeted-fraction analysis is reproduced
as written; no new metric was invented because that would change the source method.

Repeated member episodes are retained as analytical records. Every member is kept
entirely in either training or testing to prevent identity leakage. Death fields are
not available in this source file; no row is excluded based on death, and no
death-derived predictor is created.


---
## 1. Install packages if needed
---


In [ ]:
from pathlib import Path
import importlib.util
import sys

required_imports = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'xgboost': 'xgboost',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
}
missing = [pip_name for import_name, pip_name in required_imports.items() if importlib.util.find_spec(import_name) is None]
if missing:
    raise ImportError('Install missing packages with: pip install ' + ' '.join(missing))

for candidate in [Path.cwd(), Path.cwd() / 'Code']:
    if (candidate / '_prism_model_utils.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not find _prism_model_utils.py in the notebook folder or ./Code')


---
## 2. Load packages
---


In [ ]:
from itertools import product
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb

try:
    import shap
except ImportError:
    shap = None

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, ElasticNetCV, LogisticRegression, LogisticRegressionCV
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display

from _prism_model_utils import (
    GITHUB_XLSX_URL,
    align_to_columns,
    assert_xgb_booster_uses_cuda,
    clean_feature_names,
    clean_names_simple,
    ensure_output_folder,
    impute_categorical,
    impute_numeric,
    make_design_matrix,
    ntile_desc,
    project_root,
    read_prism_excel,
    require_columns,
    resolve_xgb_gpu_params,
    shap_importance_frame,
    split_train_test,
    to_binary,
    xgb_importance_frame,
    xgb_training_params,
)

warnings.filterwarnings('ignore', category=ConvergenceWarning)
PROJECT_ROOT = project_root()


---
## 3. FILE PATHS
---


In [ ]:
oregon_csv_path = PROJECT_ROOT / 'DataSets' / 'Oregon_2YearDataset.csv'
if not oregon_csv_path.exists():
    nested_candidate = PROJECT_ROOT / 'prism_repo' / 'DataSets' / 'Oregon_2YearDataset.csv'
    if nested_candidate.exists():
        oregon_csv_path = nested_candidate
    else:
        raise FileNotFoundError(
            'Could not locate Oregon_2YearDataset.csv in either expected repository location.'
        )

output_root = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'Uplift_Oregon')
output_folder = ensure_output_folder(output_root / 'Python')
r_output_folder = ensure_output_folder(output_root / 'R')
tlearner_root_folder = ensure_output_folder(output_folder / 'T-Learner')
xgboost_output_folder = ensure_output_folder(tlearner_root_folder / 'XGBoost')
glmnet_output_folder = ensure_output_folder(tlearner_root_folder / 'GLMNet')

xgboost_output_path = xgboost_output_folder / 'uplift_scored_output.csv'
xgboost_summary_path = xgboost_output_folder / 'uplift_decile_summary.csv'
glmnet_output_path = glmnet_output_folder / 'uplift_scored_output.csv'
glmnet_summary_path = glmnet_output_folder / 'uplift_decile_summary.csv'
output_path = xgboost_output_path
summary_path = xgboost_summary_path

print('Oregon data path:', oregon_csv_path)
print('Project root resolved to:', PROJECT_ROOT)
print('Oregon output root:', output_root)
print('XGBoost outputs will be saved to:', xgboost_output_folder)
print('GLMNET outputs will be saved to:', glmnet_output_folder)


---
## 4. HELPER FUNCTIONS
---


In [ ]:
def safe_as_date(values):
    return pd.to_datetime(values, errors='coerce')


def present_columns(columns, df):
    return [column for column in columns if column in df.columns]


def safe_auc(y_true, y_pred, label):
    y_series = pd.Series(y_true).dropna()
    if y_series.nunique() < 2:
        print(f'{label} AUC: cannot calculate because only one outcome class is present')
        return np.nan
    auc_value = roc_auc_score(y_true, y_pred)
    print(f'{label} AUC: {auc_value:.4f}')
    return auc_value


REQUIRE_GPU_FOR_XGBOOST = True
RUN_CPU_ONLY_COMPARISON_MODELS = True
XGBOOST_CUDA_DEVICE = 0
XGB_GPU_PARAMS = resolve_xgb_gpu_params(cuda_device=XGBOOST_CUDA_DEVICE) if REQUIRE_GPU_FOR_XGBOOST else {}
print('XGBoost GPU required:', REQUIRE_GPU_FOR_XGBOOST)
print('XGBoost CUDA params used for training:', XGB_GPU_PARAMS)
print('CPU-only GLMNET comparison enabled:', RUN_CPU_ONLY_COMPARISON_MODELS)


def make_dmatrix(x_matrix, y=None):
    if y is None:
        return xgb.DMatrix(x_matrix, feature_names=list(x_matrix.columns))
    return xgb.DMatrix(x_matrix, label=np.asarray(y, dtype=float), feature_names=list(x_matrix.columns))


def fit_xgb_cv_grid(x_matrix, y, grid, nrounds_max=500, nfold=5, seed=123):
    y_array = np.asarray(y, dtype=float)
    dtrain = make_dmatrix(x_matrix, y_array)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfold, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for XGBoost CV.')

    results = []
    best_model_info = None
    best_auc = -np.inf

    for params_grid in grid:
        params_i = xgb_training_params(
            XGB_GPU_PARAMS,
            {
                'max_depth': params_grid['max_depth'],
                'eta': params_grid['eta'],
                'min_child_weight': params_grid['min_child_weight'],
                'subsample': 0.8,
                'colsample_bytree': 0.8,
            },
            eval_metric='auc',
            seed=seed,
        )
        cv_i = xgb.cv(
            params=params_i,
            dtrain=dtrain,
            num_boost_round=nrounds_max,
            nfold=folds,
            stratified=True,
            early_stopping_rounds=20,
            seed=seed,
            verbose_eval=False,
        )
        auc_column = 'test-auc-mean'
        best_iter_i = int(cv_i[auc_column].idxmax())
        best_auc_i = float(cv_i.loc[best_iter_i, auc_column])
        best_nrounds_i = best_iter_i + 1
        results.append({
            'max_depth': params_grid['max_depth'],
            'eta': params_grid['eta'],
            'min_child_weight': params_grid['min_child_weight'],
            'best_nrounds': best_nrounds_i,
            'cv_auc': best_auc_i,
        })
        if best_auc_i > best_auc:
            best_auc = best_auc_i
            best_model_info = {'params': params_i, 'best_nrounds': best_nrounds_i, 'cv_auc': best_auc_i}

    search_results = pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True)
    final_model = xgb.train(
        params=best_model_info['params'],
        dtrain=dtrain,
        num_boost_round=best_model_info['best_nrounds'],
        verbose_eval=False,
    )
    return {
        'model': final_model,
        'best_params': best_model_info['params'],
        'best_nrounds': best_model_info['best_nrounds'],
        'best_cv_auc': best_model_info['cv_auc'],
        'search_results': search_results,
    }


class PrefitScaledLogisticPipeline:
    def __init__(self, scaler, model):
        self.named_steps = {
            'standardscaler': scaler,
            'logisticregressioncv': model,
        }

    def predict_proba(self, x_matrix):
        return self.named_steps['logisticregressioncv'].predict_proba(
            self.named_steps['standardscaler'].transform(x_matrix)
        )

    def predict(self, x_matrix):
        return self.named_steps['logisticregressioncv'].predict(
            self.named_steps['standardscaler'].transform(x_matrix)
        )


def fit_elastic_net(x_matrix, y, alpha=0.5, lambda_value=1.0, seed=123, prefit_scaler=None):
    y_array = np.asarray(y, dtype=float)
    if pd.Series(y_array).nunique() < 2:
        raise ValueError('Need both outcome classes to fit the fixed elastic-net model.')

    if not RUN_CPU_ONLY_COMPARISON_MODELS:
        raise RuntimeError(
            'fit_elastic_net uses sklearn LogisticRegression, which trains on CPU. '
            'Set RUN_CPU_ONLY_COMPARISON_MODELS = True to run this CPU comparison model.'
        )

    # Fixed standard elastic-net specification: no CV and no hyperparameter search.
    # sklearn uses C = 1 / lambda, so lambda=1 corresponds to its default C=1.
    model = LogisticRegression(
        C=float(1 / lambda_value),
        penalty='elasticnet',
        solver='saga',
        l1_ratio=float(alpha),
        max_iter=2000,
        tol=1e-3,
        random_state=seed,
    )

    if prefit_scaler is None:
        scaler = StandardScaler().fit(x_matrix)
    else:
        scaler = prefit_scaler
    model.fit(scaler.transform(x_matrix), y_array)
    pipeline = PrefitScaledLogisticPipeline(scaler, model)
    fixed_specification = pd.DataFrame([{
        'alpha': float(alpha),
        'lambda': float(lambda_value),
        'cv_auc': np.nan,
        'selection_method': 'fixed_no_tuning',
    }])
    return {
        'best_model': pipeline,
        'best_alpha': float(alpha),
        'best_lambda': float(lambda_value),
        'best_auc': np.nan,
        'search_results': fixed_specification,
    }


def build_uplift_results(base_df, pred_treated, pred_control):
    results = base_df.copy()
    results['pred_ed_if_treated'] = pred_treated
    results['pred_ed_if_control'] = pred_control
    results['benefit_score'] = results['pred_ed_if_control'] - results['pred_ed_if_treated']
    results['uplift_bad_outcome'] = results['pred_ed_if_treated'] - results['pred_ed_if_control']
    results['uplift_decile'] = ntile_desc(results['benefit_score'], 10).to_numpy()
    return results


def summarize_uplift_deciles(results):
    return (
        results
        .groupby('uplift_decile', as_index=False)
        .agg(
            n=('outcome_ed_90d', 'size'),
            avg_benefit_score=('benefit_score', 'mean'),
            observed_ed_rate=('outcome_ed_90d', 'mean'),
            treated_pct=('intervention_flag', 'mean'),
            avg_pred_ed_if_treated=('pred_ed_if_treated', 'mean'),
            avg_pred_ed_if_control=('pred_ed_if_control', 'mean'),
        )
        .sort_values('uplift_decile')
    )


def print_highest_benefit(results, label, n=20):
    print(f'{label} top {n} highest-benefit members:')
    display(
        results
        .sort_values('benefit_score', ascending=False)
        [[
            'outcome_ed_90d',
            'intervention_flag',
            'pred_ed_if_treated',
            'pred_ed_if_control',
            'benefit_score',
            'uplift_decile',
        ]]
        .head(n)
    )
    print()


def glmnet_contribution_importance_frame(model_info, x_matrix, label):
    pipeline = model_info['best_model']
    scaler = pipeline.named_steps['standardscaler']
    glmnet_model = pipeline.named_steps['logisticregressioncv']
    x_scaled = scaler.transform(x_matrix)
    coefficients = glmnet_model.coef_.ravel()
    contributions = x_scaled * coefficients
    return (
        pd.DataFrame(
            {
                'feature': list(x_matrix.columns),
                'mean_abs_model_contribution': np.abs(contributions).mean(axis=0),
                'coefficient': coefficients,
                'model': label,
                'importance_type': 'standardized_logit_contribution',
            }
        )
        .sort_values('mean_abs_model_contribution', ascending=False)
        .reset_index(drop=True)
    )



---
## 5. READ EXCEL FILE
---


In [ ]:
df_raw = pd.read_csv(oregon_csv_path, low_memory=False)
original_column_names = list(df_raw.columns)
df_raw.columns = clean_names_simple(df_raw.columns)
normalized_column_names = list(df_raw.columns)
if pd.Index(df_raw.columns).duplicated().any():
    duplicates = pd.Index(df_raw.columns)[pd.Index(df_raw.columns).duplicated()].tolist()
    raise ValueError(f'Duplicate columns after normalization: {duplicates}')

original_to_normalized = pd.DataFrame({
    'original_oregon_column': original_column_names,
    'normalized_column': normalized_column_names,
})

print('Rows:', len(df_raw))
print('Columns:', len(df_raw.columns))
print('Original-to-normalized columns:')
display(original_to_normalized)


---
## 6. CHECK REQUIRED COLUMNS
---


In [ ]:
required_fields = ['member_id', 'outcome_ed_90d', 'engaged_flag', 'optout_flag']
require_columns(df_raw, required_fields)


---
## 7. BASIC CLEANUP
---


In [ ]:
df = df_raw.copy()
raw_row_count = len(df)
unique_member_count = int(df['member_id'].nunique(dropna=True))
duplicate_member_count = int(df['member_id'].duplicated().sum())
exact_duplicate_row_count = int(df.duplicated().sum())

source_engaged = to_binary(df['engaged_flag'])
source_optout = to_binary(df['optout_flag'])
source_engaged = source_engaged.where(source_engaged.isin([0.0, 1.0]))
source_optout = source_optout.where(source_optout.isin([0.0, 1.0]))

resolved_treated = source_engaged.eq(1.0) & source_optout.eq(0.0)
resolved_control = source_engaged.eq(0.0) & source_optout.eq(1.0)
both_present = source_engaged.notna() & source_optout.notna()
contradictory_treatment = both_present & ~(resolved_treated | resolved_control)

df['source_engaged_flag'] = source_engaged
df['optout_flag'] = source_optout
df['intervention_flag'] = np.select(
    [resolved_treated, resolved_control], [1.0, 0.0], default=np.nan
)
df['treatment_derivation_status'] = np.select(
    [contradictory_treatment, resolved_treated, resolved_control],
    ['contradictory', 'treated', 'control'],
    default='unresolved',
)

outcome_numeric = pd.to_numeric(df['outcome_ed_90d'], errors='coerce')
missing_outcome_before_fill = int(outcome_numeric.isna().sum())
if (outcome_numeric.dropna() < 0).any():
    raise ValueError('outcome_ed_90d contains negative values.')
raw_outcome_count_distribution = outcome_numeric.fillna(0.0).value_counts().sort_index().to_dict()
positive_counts_binarized = int((outcome_numeric.fillna(0.0) > 0).sum())
df['outcome_ed_90d'] = (outcome_numeric.fillna(0.0) > 0).astype(float)

# ProgramFinal is the Oregon schema counterpart to the Funds program field.
# It is copied only into the standardized analysis frame; the source CSV is unchanged.
if 'programfinal' in df.columns and 'program' not in df.columns:
    df['program'] = df['programfinal']

risk_numeric = pd.to_numeric(df['risk_tier'], errors='coerce')
missing_risk_tier_rows = int(risk_numeric.isna().sum())
invalid_risk_tier_rows = int((risk_numeric.notna() & ~risk_numeric.isin([1, 2, 3, 4, 5])).sum())
valid_risk_tier = risk_numeric.where(risk_numeric.isin([1, 2, 3, 4, 5]))
df['risk_tier'] = valid_risk_tier.astype('Int64').astype('string').fillna('Missing')

death_related_fields_found = [
    column for column in df_raw.columns
    if 'death' in canonical_column_name(column) or 'deceased' in canonical_column_name(column)
] if 'canonical_column_name' in globals() else [
    column for column in df_raw.columns
    if 'death' in str(column).lower() or 'deceased' in str(column).lower()
]
rows_with_available_death_information = 0
if death_related_fields_found:
    death_available_mask = pd.Series(False, index=df_raw.index)
    for column in death_related_fields_found:
        values = df_raw[column].astype('string').str.strip()
        death_available_mask |= values.notna() & values.ne('')
    rows_with_available_death_information = int(death_available_mask.sum())

print('Engaged_flag versus OptOut_flag validation:')
display(pd.crosstab(
    df['source_engaged_flag'].fillna('Missing'),
    df['optout_flag'].fillna('Missing'),
    dropna=False,
))
print('Treatment derivation status:')
display(df['treatment_derivation_status'].value_counts(dropna=False).rename_axis('status').to_frame('n'))
print('Missing outcomes filled with zero:', missing_outcome_before_fill)
print('Records with one or more ED visits after binarization:', positive_counts_binarized)
print('Unique members:', unique_member_count)
print('Repeated member episodes retained:', duplicate_member_count)
print('Exact duplicate rows:', exact_duplicate_row_count)
print('Oregon risk-tier distribution (1 = lowest, 5 = highest):')
display(df['risk_tier'].value_counts(dropna=False).reindex(['1', '2', '3', '4', '5', 'Missing'], fill_value=0))
print('Missing source risk tiers:', missing_risk_tier_rows)
print('Out-of-range source risk tiers treated as unassigned:', invalid_risk_tier_rows)
print('Death-related source fields found:', death_related_fields_found)
print('Rows with available death information (reporting only):', rows_with_available_death_information)


---
## 8. DERIVE DATE FEATURES
---


In [ ]:
# No features are derived from program, engagement, intervention, outcome, or death dates.
# These are post-treatment or otherwise prohibited fields and are excluded before modeling.
print('Post-treatment and death-related feature engineering intentionally skipped for Oregon 2-Year.')


---
## 9. SELECT PREDICTORS
---


In [ ]:
import re


def canonical_column_name(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).lower())


candidate_predictors_all = [
    'client_contract', 'age', 'gender', 'dual_elg', 'county', 'state', 'plan_type',
    'diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag',
    'anxiety_flag', 'substance_use_flag', 'ckd_flag', 'bh_flag', 'htn_flag',
    'hyperlipidemia_flag', 'cad_flag', 'financial_strain_flag', 'food_insecurity_flag',
    'housing_instability_flag', 'transportation_barrier_flag',
    'healthliteracy_challenges_flag', 'op_visits_last_6m', 'ed_visits_last_30d',
    'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m',
    'total_cost_last_6m', 'rx_count_last_6m', 'opioid_flag', 'polypharmacy_flag',
    'current_risk_score', 'percolator_score', 'risk_tier', 'program',
]
prohibited_names = [
    'programstart', 'programend', 'engaged_date', 'engagement_length',
    'interventioncount', 'successful_intervention', 'successfulinterventions',
    'outcome_ed_30d', 'outcome_ed_6mo', 'outcome_admit_30d', 'outcome_admit_90d',
    'outcome_admit_6mo', 'outcome_total_cost_90d', 'outcome_total_cost_6mo',
    'total_cost_6mo',
]
prohibited_canonical = {canonical_column_name(column) for column in prohibited_names}
raw_treatment_canonical = {
    canonical_column_name('source_engaged_flag'),
    canonical_column_name('engaged_flag'),
    canonical_column_name('optout_flag'),
}
target_canonical = {canonical_column_name('outcome_ed_90d')}
death_canonical = {
    canonical_column_name(column) for column in df.columns
    if 'death' in canonical_column_name(column) or 'deceased' in canonical_column_name(column)
}

candidate_predictors = [column for column in candidate_predictors_all if column in df.columns]
missing_predictors = [column for column in candidate_predictors_all if column not in df.columns]
exclusion_reasons = {}
for column in df.columns:
    canonical = canonical_column_name(column)
    if column in candidate_predictors:
        continue
    if (
        canonical in prohibited_canonical
        or (column.startswith('outcome_') and column != 'outcome_ed_90d')
        or canonical in death_canonical
    ):
        exclusion_reasons[column] = 'post-treatment, death-related, alternate outcome, or explicitly prohibited'
    elif canonical in raw_treatment_canonical or column == 'intervention_flag':
        exclusion_reasons[column] = 'treatment definition/source; retained as metadata but excluded from predictors'
    elif column == 'member_id':
        exclusion_reasons[column] = 'identifier retained in outputs but excluded from predictors'
    elif column == 'treatment_derivation_status':
        exclusion_reasons[column] = 'treatment derivation audit metadata'
    elif column == 'programfinal':
        exclusion_reasons[column] = 'source column standardized to program; source duplicate excluded'
    else:
        exclusion_reasons[column] = 'not on leakage-safe baseline predictor allowlist'

meta_columns = [
    'member_id', 'source_engaged_flag', 'optout_flag',
    'treatment_derivation_status', 'outcome_ed_90d', 'intervention_flag',
]
model_df = df[[*meta_columns, *candidate_predictors]].copy()
unresolved_mask = ~model_df['treatment_derivation_status'].isin(['treated', 'control'])
excluded_treatment_rows = int(unresolved_mask.sum())
model_df = model_df.loc[~unresolved_mask].copy().reset_index(drop=True)

found_post_treatment_fields = [
    column for column in df_raw.columns
    if canonical_column_name(column) in prohibited_canonical
    or (column.startswith('outcome_') and column != 'outcome_ed_90d')
]

if missing_predictors:
    print('Allowlisted predictors not found:', missing_predictors)
print('Rows excluded because treatment could not be resolved:', excluded_treatment_rows)
print('Post-treatment fields found and excluded:', found_post_treatment_fields)
print('Candidate baseline predictors before train-only filtering:', candidate_predictors)


---
## 10. DATA TYPE HANDLING
---


In [ ]:
model_df.columns

In [ ]:
binary_predictors = [
    'dual_elg', 'diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag',
    'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag', 'bh_flag',
    'htn_flag', 'hyperlipidemia_flag', 'cad_flag', 'financial_strain_flag',
    'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag',
    'healthliteracy_challenges_flag', 'opioid_flag', 'polypharmacy_flag',
]
for column in present_columns(binary_predictors, model_df):
    model_df[column] = to_binary(model_df[column])

possible_numeric_cols = [
    'age', 'op_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m',
    'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m',
    'rx_count_last_6m', 'current_risk_score', 'percolator_score',
]

def parse_currency(values):
    cleaned = (
        values.astype('string')
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors='coerce')


for column in present_columns(possible_numeric_cols, model_df):
    if column == 'total_cost_last_6m':
        model_df[column] = parse_currency(model_df[column])
    else:
        model_df[column] = pd.to_numeric(model_df[column], errors='coerce')

# Missing values remain missing here. Imputation values, category levels, and
# zero-variance filtering are fitted only after the grouped train/test split.
model_df = model_df[[
    'member_id', 'source_engaged_flag', 'optout_flag', 'treatment_derivation_status',
    'outcome_ed_90d', 'intervention_flag', *candidate_predictors,
]].copy()

print('Pre-split modeling columns (missing values intentionally retained):')
print(list(model_df.columns))


---
## 11. TRAIN / TEST SPLIT
---


In [ ]:
def grouped_stratified_train_test_split(
    frame,
    group_column='member_id',
    train_fraction=0.70,
    seed=123,
    n_candidates=500,
):
    strata = frame[['intervention_flag', 'outcome_ed_90d']].astype(int).astype(str).agg('_'.join, axis=1)
    overall_distribution = strata.value_counts(normalize=True).sort_index()
    splitter = GroupShuffleSplit(
        n_splits=n_candidates,
        train_size=train_fraction,
        random_state=seed,
    )
    best = None
    best_score = np.inf
    all_strata = set(overall_distribution.index)

    for train_positions, test_positions in splitter.split(frame, strata, groups=frame[group_column]):
        train_strata = strata.iloc[train_positions]
        test_strata = strata.iloc[test_positions]
        if set(train_strata.unique()) != all_strata or set(test_strata.unique()) != all_strata:
            continue
        train_distribution = train_strata.value_counts(normalize=True).reindex(overall_distribution.index, fill_value=0)
        test_distribution = test_strata.value_counts(normalize=True).reindex(overall_distribution.index, fill_value=0)
        size_error = abs((len(train_positions) / len(frame)) - train_fraction)
        balance_error = (
            (train_distribution - overall_distribution).abs().sum()
            + (test_distribution - overall_distribution).abs().sum()
        )
        score = float(size_error + balance_error)
        if score < best_score:
            best_score = score
            best = (train_positions, test_positions)

    if best is None:
        raise ValueError('Could not create a member-grouped split containing all treatment/outcome strata.')

    train_positions, test_positions = best
    train = frame.iloc[train_positions].copy().reset_index(drop=True)
    test = frame.iloc[test_positions].copy().reset_index(drop=True)
    overlap = set(train[group_column]).intersection(test[group_column])
    if overlap:
        raise AssertionError(f'Member leakage across train/test: {sorted(overlap)[:10]}')
    return train, test, best_score


train_df, test_df, split_balance_score = grouped_stratified_train_test_split(
    model_df,
    group_column='member_id',
    train_fraction=0.70,
    seed=123,
)

train_test_member_overlap = set(train_df['member_id']).intersection(test_df['member_id'])
assert not train_test_member_overlap

print('Training records:', len(train_df))
print('Testing records:', len(test_df))
print('Training unique members:', train_df['member_id'].nunique())
print('Testing unique members:', test_df['member_id'].nunique())
print('Train/test member overlap:', len(train_test_member_overlap))
print('Grouped split balance score:', round(split_balance_score, 6))
display(
    model_df.groupby(['intervention_flag', 'outcome_ed_90d'], dropna=False)
    .size().rename('n').reset_index()
)


---
## 12. SEPARATE TREATED / CONTROL
---


In [ ]:
train_treated = train_df[train_df['intervention_flag'] == 1].copy()
train_control = train_df[train_df['intervention_flag'] == 0].copy()

print('Training treated rows:', len(train_treated))
print('Training control rows:', len(train_control))
print()

if len(train_treated) < 50:
    raise ValueError('Too few treated rows to train a stable model.')
if len(train_control) < 50:
    raise ValueError('Too few control rows to train a stable model.')


---
## 13. BUILD MODEL MATRICES
---


In [ ]:
import json

id_cols = ['member_id', 'source_engaged_flag', 'optout_flag', 'treatment_derivation_status']

# Feature selection, imputation statistics, and category levels are learned from
# training records only. This prevents information from the held-out set from
# influencing preprocessing.
train_unique_counts = train_df[candidate_predictors].nunique(dropna=True)
zero_variance_predictors = train_unique_counts[train_unique_counts <= 1].index.tolist()
feature_cols = [column for column in candidate_predictors if column not in zero_variance_predictors]
for column in zero_variance_predictors:
    exclusion_reasons[column] = 'zero-variance or all-missing predictor removed using training data only'

numeric_feature_cols = [
    column for column in feature_cols
    if column in possible_numeric_cols or column in binary_predictors
]
categorical_feature_cols = [column for column in feature_cols if column not in numeric_feature_cols]

train_numeric_medians = {}
for column in numeric_feature_cols:
    values = pd.to_numeric(train_df[column], errors='coerce')
    median = values.median(skipna=True)
    train_numeric_medians[column] = 0.0 if pd.isna(median) else float(median)

train_category_levels = {}
for column in categorical_feature_cols:
    values = train_df[column].astype('string')
    values = values.where(values.notna() & values.str.strip().ne(''), 'Missing')
    levels = sorted(values.astype(str).unique().tolist())
    if '__Unseen__' not in levels:
        levels.append('__Unseen__')
    train_category_levels[column] = levels


def prepare_predictor_frame(frame):
    prepared = pd.DataFrame(index=frame.index)
    for column in numeric_feature_cols:
        values = pd.to_numeric(frame[column], errors='coerce')
        prepared[column] = values.fillna(train_numeric_medians[column]).astype(float)
    for column in categorical_feature_cols:
        values = frame[column].astype('string')
        values = values.where(values.notna() & values.str.strip().ne(''), 'Missing').astype(str)
        known = set(train_category_levels[column])
        values = values.where(values.isin(known), '__Unseen__')
        prepared[column] = pd.Categorical(values, categories=train_category_levels[column])
    return prepared[feature_cols]


def design_matrix_from_prepared(prepared, expected_columns=None):
    matrix = pd.get_dummies(prepared, drop_first=False, dtype=float)
    matrix.columns = clean_feature_names(matrix.columns)
    matrix = matrix.astype(float)
    if expected_columns is not None:
        matrix = align_to_columns(matrix, expected_columns)
    return matrix


train_prepared_df = prepare_predictor_frame(train_df)
test_prepared_df = prepare_predictor_frame(test_df)
train_matrix = design_matrix_from_prepared(train_prepared_df)
test_matrix = design_matrix_from_prepared(test_prepared_df, train_matrix.columns)

train_treated_mask = train_df['intervention_flag'].eq(1).to_numpy()
train_control_mask = train_df['intervention_flag'].eq(0).to_numpy()
x_treated = train_matrix.iloc[train_treated_mask].reset_index(drop=True)
x_control = train_matrix.iloc[train_control_mask].reset_index(drop=True)
x_test = test_matrix.reset_index(drop=True)
combined_matrix = train_matrix.copy()
y_treated = train_df.loc[train_treated_mask, 'outcome_ed_90d'].astype(float).to_numpy()
y_control = train_df.loc[train_control_mask, 'outcome_ed_90d'].astype(float).to_numpy()

forbidden_feature_canonical = (
    prohibited_canonical
    | raw_treatment_canonical
    | target_canonical
    | death_canonical
    | {canonical_column_name('intervention_flag')}
)

def is_forbidden_feature(column):
    canonical = canonical_column_name(column)
    if 'death' in canonical or 'deceased' in canonical:
        return True
    return any(canonical == item or canonical.startswith(item) for item in forbidden_feature_canonical)


leaking_features = [column for column in feature_cols if is_forbidden_feature(column)]
leaking_matrix_columns = [column for column in combined_matrix.columns if is_forbidden_feature(column)]
assert not leaking_features, f'Prohibited predictors found: {leaking_features}'
assert not leaking_matrix_columns, f'Encoded leakage columns found: {leaking_matrix_columns}'
assert set(pd.Series(y_treated).unique()).issubset({0.0, 1.0})
assert set(pd.Series(y_control).unique()).issubset({0.0, 1.0})
assert 'member_id' not in feature_cols
assert not set(train_df['member_id']).intersection(test_df['member_id'])
assert len(model_df) == raw_row_count - excluded_treatment_rows
assert int((df['outcome_ed_90d'] == 1).sum()) == positive_counts_binarized

split_distribution = pd.concat([
    train_df.assign(split='train'), test_df.assign(split='test')
]).groupby(['split', 'intervention_flag', 'outcome_ed_90d'], dropna=False).size().rename('n').reset_index()
split_distribution.to_csv(output_folder / 'preprocessing_split_distribution.csv', index=False)

transformation_lookup = {
    'member_id': 'Retained as output identifier and grouping key; excluded from predictors.',
    'engaged_flag': 'Converted to binary and used as the canonical treatment source only.',
    'optout_flag': 'Converted to binary and used only to validate inverse treatment coding.',
    'outcome_ed_90d': 'Converted to numeric; missing set to 0; values greater than 0 mapped to 1.',
    'programfinal': 'Standardized to program; removed because it has no training variation.',
    'risk_tier': 'Values 1-5 retained as categories; missing or out-of-range values mapped to Missing.',
    'total_cost_last_6m': 'Currency symbols and commas removed; parsed as numeric; train-median imputation.',
}

column_audit_rows = []
for original_name, normalized_name in zip(original_column_names, normalized_column_names):
    modeling_name = 'program' if normalized_name == 'programfinal' else normalized_name
    canonical = canonical_column_name(normalized_name)
    if normalized_name == 'engaged_flag':
        role = 'treatment source'
        status = 'excluded'
        reason = 'canonical treatment source; never a predictor'
    elif normalized_name == 'optout_flag':
        role = 'treatment validation source'
        status = 'excluded'
        reason = 'redundant inverse treatment signal; validation only'
    elif normalized_name == 'outcome_ed_90d':
        role = 'outcome'
        status = 'excluded'
        reason = 'binary target; never a predictor'
    elif normalized_name == 'member_id':
        role = 'identifier/grouping key'
        status = 'excluded'
        reason = 'retained for grouped splitting and outputs; never a predictor'
    elif normalized_name == 'case_manager_name':
        role = 'excluded identifier/all-missing field'
        status = 'excluded'
        reason = 'non-generalizable identifier field and entirely missing in Oregon'
    elif (
        canonical in prohibited_canonical
        or (normalized_name.startswith('outcome_') and normalized_name != 'outcome_ed_90d')
    ):
        role = 'post-treatment variable'
        status = 'excluded'
        reason = 'post-treatment leakage control'
    elif 'death' in canonical or 'deceased' in canonical:
        role = 'death-related variable'
        status = 'excluded'
        reason = 'death information is not used or transformed into a predictor'
    elif modeling_name in feature_cols:
        role = 'eligible baseline predictor'
        status = 'included'
        reason = 'leakage-safe baseline predictor retained after train-only filtering'
    elif modeling_name in zero_variance_predictors:
        role = 'candidate baseline predictor'
        status = 'excluded'
        reason = exclusion_reasons.get(modeling_name, 'zero variance in training data')
    else:
        role = 'excluded field'
        status = 'excluded'
        reason = exclusion_reasons.get(normalized_name, 'not on leakage-safe baseline predictor allowlist')

    if normalized_name in transformation_lookup:
        transformation = transformation_lookup[normalized_name]
    elif modeling_name in numeric_feature_cols:
        transformation = 'Numeric coercion and train-median imputation.'
    elif modeling_name in categorical_feature_cols:
        transformation = 'Missing category handling and one-hot encoding using training levels only.'
    else:
        transformation = 'No modeling transformation; field excluded before preprocessing.'

    column_audit_rows.append({
        'original_oregon_column': original_name,
        'normalized_column': normalized_name,
        'modeling_name': modeling_name,
        'role': role,
        'status': status,
        'transformation': transformation,
        'reason': reason,
    })

preprocessing_column_audit = pd.DataFrame(column_audit_rows)
preprocessing_column_audit.to_csv(output_folder / 'preprocessing_column_audit.csv', index=False)

preprocessing_audit_summary = pd.DataFrame([{
    'raw_rows': raw_row_count,
    'raw_columns': len(df_raw.columns),
    'unique_members': unique_member_count,
    'repeated_member_episodes_retained': duplicate_member_count,
    'exact_duplicate_rows': exact_duplicate_row_count,
    'missing_outcome_filled_with_zero': missing_outcome_before_fill,
    'positive_outcome_counts_binarized_to_one': positive_counts_binarized,
    'contradictory_treatment_rows': int((df['treatment_derivation_status'] == 'contradictory').sum()),
    'unresolved_treatment_rows': int((df['treatment_derivation_status'] == 'unresolved').sum()),
    'modeling_rows': len(model_df),
    'treated_rows': int((model_df['intervention_flag'] == 1).sum()),
    'control_rows': int((model_df['intervention_flag'] == 0).sum()),
    'outcome_positive_rows': int((model_df['outcome_ed_90d'] == 1).sum()),
    'outcome_zero_rows': int((model_df['outcome_ed_90d'] == 0).sum()),
    'death_related_fields_found': '; '.join(death_related_fields_found),
    'rows_with_available_death_information': rows_with_available_death_information,
    'rows_excluded_for_death': 0,
    'missing_source_risk_tier_rows': missing_risk_tier_rows,
    'out_of_range_source_risk_tier_rows': invalid_risk_tier_rows,
    'train_rows': len(train_df),
    'test_rows': len(test_df),
    'train_unique_members': train_df['member_id'].nunique(),
    'test_unique_members': test_df['member_id'].nunique(),
    'train_test_member_overlap': len(set(train_df['member_id']).intersection(test_df['member_id'])),
    'retained_predictor_count': len(feature_cols),
    'encoded_feature_count': combined_matrix.shape[1],
    'zero_variance_predictors_removed': '; '.join(zero_variance_predictors),
    'retained_predictors': '; '.join(feature_cols),
    'post_treatment_fields_found': '; '.join(found_post_treatment_fields),
    'treatment_definition': 'Engaged_flag=1 treated; Engaged_flag=0 control; validated as inverse of OptOut_flag',
    'treatment_distribution': json.dumps(model_df['intervention_flag'].value_counts().sort_index().to_dict()),
    'binary_outcome_distribution': json.dumps(model_df['outcome_ed_90d'].value_counts().sort_index().to_dict()),
    'raw_count_distribution_before_binary': json.dumps(raw_outcome_count_distribution),
    'preprocessing_fit_on_training_only': True,
    'grouped_split_by_member_id': True,
    'leakage_assertions_passed': True,
    'xgboost_execution_backend': 'CUDA GPU required on device 0; modeling grid unchanged',
}])
preprocessing_audit_summary.to_csv(output_folder / 'preprocessing_audit_summary.csv', index=False)

print('Zero-variance predictors removed using training data only:', zero_variance_predictors)
print('Preprocessing audit summary:')
display(preprocessing_audit_summary)
print('Column mapping and role audit:')
display(preprocessing_column_audit)
print('Leakage checks passed; final encoded feature count:', combined_matrix.shape[1])


---
## WRITE-UP DATA REVIEW SUMMARY
---


In [ ]:
def is_binary_indicator_column(series):
    non_missing = pd.Series(series).dropna()
    if non_missing.empty:
        return False
    numeric_values = pd.to_numeric(non_missing, errors='coerce')
    if numeric_values.isna().any():
        return False
    return set(numeric_values.unique()).issubset({0, 1}) and numeric_values.nunique() <= 2


model_feature_type_rows = []
for column in feature_cols:
    series = model_df[column]
    if is_binary_indicator_column(series):
        predictor_type = 'Binary indicator'
    elif column in possible_numeric_cols or pd.api.types.is_numeric_dtype(series):
        predictor_type = 'Continuous/count numeric'
    else:
        predictor_type = 'Multi-level categorical'
    model_feature_type_rows.append({'variable': column, 'predictor_type': predictor_type})

model_feature_type_summary = pd.DataFrame(model_feature_type_rows)
predictor_type_counts = model_feature_type_summary['predictor_type'].value_counts().to_dict()

continuous_count_predictors = int(predictor_type_counts.get('Continuous/count numeric', 0))
binary_indicator_predictors = int(predictor_type_counts.get('Binary indicator', 0))
multilevel_categorical_predictors = int(predictor_type_counts.get('Multi-level categorical', 0))

data_review_summary = pd.DataFrame([{
    'total_analytical_records': len(model_df),
    'unique_members': model_df['member_id'].nunique(),
    'treated_records': int((model_df['intervention_flag'] == 1).sum()),
    'control_records': int((model_df['intervention_flag'] == 0).sum()),
    'treatment_rate': float(model_df['intervention_flag'].mean()),
    'outcome_events': int((model_df['outcome_ed_90d'] == 1).sum()),
    'outcome_prevalence': float(model_df['outcome_ed_90d'].mean()),
    'treated_outcome_rate': float(model_df.loc[model_df['intervention_flag'] == 1, 'outcome_ed_90d'].mean()),
    'control_outcome_rate': float(model_df.loc[model_df['intervention_flag'] == 0, 'outcome_ed_90d'].mean()),
    'number_of_predictors': len(feature_cols),
    'number_of_continuous_count_predictors': continuous_count_predictors,
    'number_of_binary_indicator_predictors': binary_indicator_predictors,
    'number_of_multilevel_categorical_predictors': multilevel_categorical_predictors,
    'number_of_model_matrix_columns': x_test.shape[1],
}])

data_review_summary_path = output_folder / 'data_review_summary.csv'
data_review_summary.to_csv(data_review_summary_path, index=False)

print('Data review summary:')
display(data_review_summary)
print('Data review summary written to:', data_review_summary_path)


---
## WRITE-UP PREDICTOR DATA DICTIONARY AND SUMMARY
---


In [ ]:
predictor_category_lookup = {
    # Demographics
    'client_contract': 'Demographics',
    'service_region': 'Demographics',
    'program': 'Demographics',
    'case_manager_name': 'Demographics',
    'age': 'Demographics',
    'gender': 'Demographics',
    'dual_eligible': 'Demographics',
    'dual_elg': 'Demographics',
    'state': 'Demographics',
    'county': 'Demographics',
    'plan_type': 'Demographics',
    'language': 'Demographics',
    'living_alone_flag': 'Demographics',

    # Clinical Conditions
    'diabetes_flag': 'Clinical Conditions',
    'chf_flag': 'Clinical Conditions',
    'copd_flag': 'Clinical Conditions',
    'asthma_flag': 'Clinical Conditions',
    'depression_flag': 'Clinical Conditions',
    'anxiety_flag': 'Clinical Conditions',
    'substance_use_flag': 'Clinical Conditions',
    'ckd_flag': 'Clinical Conditions',
    'pregnancy_flag': 'Clinical Conditions',
    'behavioral_health_risk_flag': 'Clinical Conditions',
    'bh_flag': 'Clinical Conditions',
    'htn_flag': 'Clinical Conditions',
    'hyperlipidemia_flag': 'Clinical Conditions',
    'cad_flag': 'Clinical Conditions',

    # SDOH
    'food_insecurity_flag': 'SDOH',
    'housing_instability_flag': 'SDOH',
    'transportation_barrier_flag': 'SDOH',
    'utilities_insecurity_flag': 'SDOH',
    'financial_strain_flag': 'SDOH',
    'healthliteracy_challenges_flag': 'SDOH',

    # Utilization
    'pcp_visits_last_6m': 'Utilization',
    'op_visits_last_6m': 'Utilization',
    'specialist_visits_last_6m': 'Utilization',
    'ed_visits_last_30d': 'Utilization',
    'ed_visits_last_6m': 'Utilization',
    'admits_last_6m': 'Utilization',
    'observation_stays_last_6m': 'Utilization',

    # Pharmacy
    'total_cost_last_6m': 'Pharmacy',
    'rx_count_last_6m': 'Pharmacy',
    'med_adherence_pdc': 'Pharmacy',
    'high_cost_drug_flag': 'Pharmacy',
    'opioid_flag': 'Pharmacy',
    'polypharmacy_flag': 'Pharmacy',

    # Risk Scores
    'utilization_score': 'Risk Scores',
    'clinical_score': 'Risk Scores',
    'sdoh_score': 'Risk Scores',
    'percolator_utilization_score': 'Risk Scores',
    'percolator_clinical_score': 'Risk Scores',
    'percolator_sdoh_score': 'Risk Scores',
    'current_risk_score': 'Risk Scores',
    'percolator_score': 'Risk Scores',
    'risk_tier': 'Risk Scores',
}

predictor_description_lookup = {
    'client_contract': 'Client contract or line of business associated with the member.',
    'service_region': 'Service region where the member receives care management support.',
    'program': 'Care management program or operational program assignment.',
    'case_manager_name': 'Assigned case manager identifier/name.',
    'age': 'Member age at the index point.',
    'gender': 'Member gender category.',
    'dual_eligible': 'Indicator or category for Medicare/Medicaid dual eligibility.',
    'dual_elg': 'Indicator for Medicare/Medicaid dual eligibility.',
    'state': 'Member state of residence.',
    'county': 'Member county of residence.',
    'plan_type': 'Health plan product or plan type.',
    'language': 'Preferred or primary language.',
    'living_alone_flag': 'Indicator that the member lives alone.',
    'diabetes_flag': 'Indicator for diabetes diagnosis/history.',
    'chf_flag': 'Indicator for congestive heart failure diagnosis/history.',
    'copd_flag': 'Indicator for COPD diagnosis/history.',
    'asthma_flag': 'Indicator for asthma diagnosis/history.',
    'depression_flag': 'Indicator for depression diagnosis/history.',
    'anxiety_flag': 'Indicator for anxiety diagnosis/history.',
    'substance_use_flag': 'Indicator for substance use-related risk or diagnosis.',
    'ckd_flag': 'Indicator for chronic kidney disease diagnosis/history.',
    'pregnancy_flag': 'Indicator for pregnancy status/history.',
    'behavioral_health_risk_flag': 'Indicator for behavioral health risk.',
    'bh_flag': 'Indicator for behavioral health history or risk.',
    'htn_flag': 'Indicator for hypertension history.',
    'hyperlipidemia_flag': 'Indicator for hyperlipidemia history.',
    'cad_flag': 'Indicator for coronary artery disease history.',
    'food_insecurity_flag': 'Indicator for food insecurity.',
    'housing_instability_flag': 'Indicator for housing instability.',
    'transportation_barrier_flag': 'Indicator for transportation barriers.',
    'utilities_insecurity_flag': 'Indicator for utility insecurity.',
    'financial_strain_flag': 'Indicator for financial strain.',
    'healthliteracy_challenges_flag': 'Indicator for health-literacy challenges.',
    'pcp_visits_last_6m': 'Number of primary care visits in the last 6 months.',
    'op_visits_last_6m': 'Number of outpatient visits in the last 6 months.',
    'specialist_visits_last_6m': 'Number of specialist visits in the last 6 months.',
    'ed_visits_last_30d': 'Number of ED visits in the last 30 days.',
    'ed_visits_last_6m': 'Number of ED visits in the last 6 months.',
    'admits_last_6m': 'Number of inpatient admissions in the last 6 months.',
    'observation_stays_last_6m': 'Number of observation stays in the last 6 months.',
    'total_cost_last_6m': 'Total healthcare cost in the last 6 months.',
    'rx_count_last_6m': 'Number of prescriptions or medication fills in the last 6 months.',
    'med_adherence_pdc': 'Medication adherence proportion of days covered.',
    'high_cost_drug_flag': 'Indicator for high-cost drug use.',
    'opioid_flag': 'Indicator for opioid medication use.',
    'polypharmacy_flag': 'Indicator for polypharmacy risk.',
    'percolator_utilization_score': 'Composite utilization risk score.',
    'percolator_clinical_score': 'Composite clinical risk score.',
    'percolator_sdoh_score': 'Composite social determinants of health risk score.',
    'current_risk_score': 'Current overall risk score.',
    'percolator_score': 'Baseline Percolator risk score.',
    'risk_tier': 'Categorical risk tier assignment.',
    'intervention_type': 'Type of intervention assigned or delivered.',
    'intervention_days_active': 'Number of days the intervention was active.',
    'touches_per_month': 'Average number of care management touches per month.',
    'outreach_attempts': 'Number of outreach attempts.',
    'successful_contacts': 'Number of successful contacts.',
    'avg_call_duration_min': 'Average call duration in minutes.',
    'max_call_duration_min': 'Maximum call duration in minutes.',
    'notes_escalation_flag': 'Indicator for escalation noted in care management documentation.',
    'community_referral_flag': 'Indicator for community referral.',
    'pharmacy_review_flag': 'Indicator for pharmacy review.',
    'engagement_level': 'Member engagement level category.',
    'days_to_intervention_start': 'Days between index date and intervention start.',
    'intervention_start_month': 'Calendar month of intervention start.',
    'intervention_start_wday': 'Day of week of intervention start.',
}


def predictor_category(column):
    return predictor_category_lookup.get(column, 'Other / derived')


def predictor_description(column):
    return predictor_description_lookup.get(column, f'Model predictor derived from `{column}`.')


def compact_examples(series, max_items=5):
    values = series.dropna().astype(str).drop_duplicates().head(max_items).tolist()
    return '; '.join(values)


def top_value_summary(series, max_items=5):
    counts = series.dropna().astype(str).value_counts().head(max_items)
    return '; '.join(f'{idx} ({count})' for idx, count in counts.items())


def is_binary_predictor(column, series):
    if column not in feature_cols:
        return False
    non_missing = pd.Series(series).dropna()
    if non_missing.empty:
        return False
    numeric_values = pd.to_numeric(non_missing, errors='coerce')
    if numeric_values.isna().any():
        return False
    return set(numeric_values.unique()).issubset({0, 1}) and numeric_values.nunique() <= 2


def predictor_type(column, series):
    if is_binary_predictor(column, series):
        return 'Binary indicator'
    if column in possible_numeric_cols:
        return 'Continuous/count numeric'
    return 'Multi-level categorical'


predictor_distribution_folder = ensure_output_folder(output_folder / 'Predictor_Distributions')
numeric_distribution_folder = ensure_output_folder(predictor_distribution_folder / 'Numeric')
categorical_distribution_folder = ensure_output_folder(predictor_distribution_folder / 'Categorical')

predictor_dictionary_rows = []
for column in candidate_predictors:
    raw_series = df[column] if column in df.columns else pd.Series(dtype='object')
    model_series = model_df[column] if column in model_df.columns else raw_series
    numeric_values = parse_currency(raw_series) if column == 'total_cost_last_6m' else pd.to_numeric(raw_series, errors='coerce')
    is_numeric_like = pd.api.types.is_numeric_dtype(model_series) or numeric_values.notna().sum() > 0
    model_matrix_matches = [matrix_col for matrix_col in x_test.columns if matrix_col == column or matrix_col.startswith(f'{column}_')]
    predictor_dictionary_rows.append(
        {
            'variable': column,
            'category': predictor_category(column),
            'data_type': str(raw_series.dtype),
            'model_ready_data_type': str(model_series.dtype),
            'description': predictor_description(column),
            'predictor_type': predictor_type(column, model_series),
            'included_in_model': column in feature_cols,
            'included_in_model_matrix': len(model_matrix_matches) > 0,
            'model_matrix_columns': '; '.join(model_matrix_matches[:20]),
            'missing_count': int(raw_series.isna().sum()),
            'missing_pct': float(raw_series.isna().mean()),
            'unique_values': int(raw_series.dropna().nunique()),
            'example_values': compact_examples(raw_series),
            'min_value': float(numeric_values.min()) if is_numeric_like and numeric_values.notna().any() else np.nan,
            'max_value': float(numeric_values.max()) if is_numeric_like and numeric_values.notna().any() else np.nan,
        }
    )

predictor_data_dictionary = pd.DataFrame(predictor_dictionary_rows).sort_values(['category', 'variable']).reset_index(drop=True)
predictor_data_dictionary_path = predictor_distribution_folder / 'predictor_data_dictionary.csv'
predictor_data_dictionary.to_csv(predictor_data_dictionary_path, index=False)

numeric_summary_rows = []
categorical_summary_rows = []
for column in candidate_predictors:
    raw_series = df[column] if column in df.columns else pd.Series(dtype='object')
    numeric_values = parse_currency(raw_series) if column == 'total_cost_last_6m' else pd.to_numeric(raw_series, errors='coerce')
    numeric_share = numeric_values.notna().mean() if len(raw_series) else 0
    treat_as_numeric = (
        column in possible_numeric_cols
        and not is_binary_predictor(column, raw_series)
        and numeric_values.notna().any()
    )

    if treat_as_numeric:
        numeric_summary_rows.append(
            {
                'variable': column,
                'category': predictor_category(column),
                'n': int(numeric_values.notna().sum()),
                'missing_count': int(raw_series.isna().sum()),
                'missing_pct': float(raw_series.isna().mean()),
                'mean': float(numeric_values.mean()),
                'median': float(numeric_values.median()),
                'std': float(numeric_values.std()),
                'min': float(numeric_values.min()),
                'p25': float(numeric_values.quantile(0.25)),
                'p75': float(numeric_values.quantile(0.75)),
                'max': float(numeric_values.max()),
            }
        )
    else:
        non_missing = raw_series.dropna().astype(str)
        value_counts = non_missing.value_counts()
        mode_value = value_counts.index[0] if len(value_counts) else np.nan
        mode_count = int(value_counts.iloc[0]) if len(value_counts) else 0
        categorical_summary_rows.append(
            {
                'variable': column,
                'category': predictor_category(column),
                'n': int(non_missing.shape[0]),
                'missing_count': int(raw_series.isna().sum()),
                'missing_pct': float(raw_series.isna().mean()),
                'unique_values': int(non_missing.nunique()),
                'mode': mode_value,
                'mode_count': mode_count,
                'mode_pct': float(mode_count / len(raw_series)) if len(raw_series) else np.nan,
                'top_values': top_value_summary(raw_series),
            }
        )

numeric_predictor_summary = pd.DataFrame(numeric_summary_rows).sort_values(['category', 'variable']).reset_index(drop=True)
categorical_predictor_summary = pd.DataFrame(categorical_summary_rows).sort_values(['category', 'variable']).reset_index(drop=True)

numeric_predictor_summary_path = predictor_distribution_folder / 'numeric_predictor_summary.csv'
categorical_predictor_summary_path = predictor_distribution_folder / 'categorical_predictor_summary.csv'
numeric_predictor_summary.to_csv(numeric_predictor_summary_path, index=False)
categorical_predictor_summary.to_csv(categorical_predictor_summary_path, index=False)

print('Predictor data dictionary:')
display(predictor_data_dictionary)
print('Predictor data dictionary written to:', predictor_data_dictionary_path)
print()

print('Numeric predictor summary:')
display(numeric_predictor_summary)
print('Numeric predictor summary written to:', numeric_predictor_summary_path)
print()

print('Categorical predictor summary:')
display(categorical_predictor_summary)
print('Categorical predictor summary written to:', categorical_predictor_summary_path)



---
## PREDICTOR DISTRIBUTION VISUALS
---


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import re


numeric_distribution_pdf_path = predictor_distribution_folder / 'numeric_predictor_distributions.pdf'
categorical_distribution_pdf_path = predictor_distribution_folder / 'categorical_predictor_distributions.pdf'
distribution_visual_index_path = predictor_distribution_folder / 'predictor_distribution_visual_index.csv'

sns.set_theme(style='whitegrid')


def safe_filename(value):
    safe_value = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')
    return safe_value or 'unnamed'


def plot_numeric_distribution(column, folder):
    values = (parse_currency(df[column]) if column == 'total_cost_last_6m' else pd.to_numeric(df[column], errors='coerce')).dropna()
    if values.empty:
        return None

    fig, ax = plt.subplots(figsize=(8, 4.8))
    unique_count = values.nunique()
    if unique_count <= 15:
        sns.histplot(values, discrete=True, shrink=0.8, ax=ax, color='#4c78a8')
    else:
        sns.histplot(values, bins=30, kde=True, ax=ax, color='#4c78a8')
    ax.set_title(f'{column} Distribution')
    ax.set_xlabel(column)
    ax.set_ylabel('Member count')
    fig.tight_layout()

    output_path = folder / f'{safe_filename(column)}_histogram.png'
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    return fig, output_path


def plot_categorical_distribution(column, folder, top_n=20):
    values = df[column].dropna().astype(str)
    if values.empty:
        return None

    counts = values.value_counts()
    if column == 'risk_tier':
        counts = counts.reindex(['1', '2', '3', '4', '5', 'Missing'], fill_value=0)
    if len(counts) > top_n:
        counts = pd.concat([counts.head(top_n), pd.Series({'Other': counts.iloc[top_n:].sum()})])

    fig_height = max(4.8, min(12, 0.35 * len(counts) + 2.2))
    fig, ax = plt.subplots(figsize=(9, fig_height))
    plot_df = counts.reset_index()
    plot_df.columns = [column, 'member_count']
    sns.barplot(data=plot_df, x='member_count', y=column, ax=ax, color='#59a14f')
    ax.set_title(f'{column} Distribution')
    ax.set_xlabel('Member count')
    ax.set_ylabel(column)
    for patch in ax.patches:
        width = patch.get_width()
        if pd.notna(width):
            ax.annotate(
                f'{int(width):,}',
                (width, patch.get_y() + patch.get_height() / 2),
                ha='left',
                va='center',
                fontsize=8,
                xytext=(4, 0),
                textcoords='offset points',
            )
    fig.tight_layout()

    output_path = folder / f'{safe_filename(column)}_bar_chart.png'
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    return fig, output_path


distribution_visual_rows = []

numeric_columns_for_plots = numeric_predictor_summary['variable'].tolist() if not numeric_predictor_summary.empty else []
with PdfPages(numeric_distribution_pdf_path) as pdf:
    for column in numeric_columns_for_plots:
        result = plot_numeric_distribution(column, numeric_distribution_folder)
        if result is None:
            continue
        fig, output_path = result
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        distribution_visual_rows.append({
            'variable': column,
            'category': predictor_category(column),
            'visual_type': 'numeric_histogram',
            'output_file': str(output_path),
        })

categorical_columns_for_plots = categorical_predictor_summary['variable'].tolist() if not categorical_predictor_summary.empty else []
with PdfPages(categorical_distribution_pdf_path) as pdf:
    for column in categorical_columns_for_plots:
        result = plot_categorical_distribution(column, categorical_distribution_folder)
        if result is None:
            continue
        fig, output_path = result
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        distribution_visual_rows.append({
            'variable': column,
            'category': predictor_category(column),
            'visual_type': 'categorical_bar_chart',
            'output_file': str(output_path),
        })

distribution_visual_index = pd.DataFrame(distribution_visual_rows)
distribution_visual_index.to_csv(distribution_visual_index_path, index=False)

print('Predictor distribution visuals written to:', predictor_distribution_folder)
print('Numeric predictor distribution PDF:', numeric_distribution_pdf_path)
print('Categorical predictor distribution PDF:', categorical_distribution_pdf_path)
print('Distribution visual index written to:', distribution_visual_index_path)
display(distribution_visual_index)


---
## 14. TRAIN XGBOOST MODELS
---


In [ ]:
print('Unique y_treated values:')
print(np.sort(pd.unique(y_treated)))
print()

print('Unique y_control values:')
print(np.sort(pd.unique(y_control)))
print()

if not set(pd.Series(y_treated).dropna().unique()).issubset({0.0, 1.0}):
    raise ValueError('y_treated contains values other than 0 and 1.')
if not set(pd.Series(y_control).dropna().unique()).issubset({0.0, 1.0}):
    raise ValueError('y_control contains values other than 0 and 1.')

dtrain_treated = make_dmatrix(x_treated, y_treated)
dtrain_control = make_dmatrix(x_control, y_control)

params = xgb_training_params(
    XGB_GPU_PARAMS,
    {
        'max_depth': 4,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    },
    eval_metric='logloss',
    seed=123,
)

model_treated = xgb.train(params=params, dtrain=dtrain_treated, num_boost_round=150, verbose_eval=False)
model_control = xgb.train(params=params, dtrain=dtrain_control, num_boost_round=150, verbose_eval=False)
assert_xgb_booster_uses_cuda(model_treated, 'Baseline treated XGBoost model')
assert_xgb_booster_uses_cuda(model_control, 'Baseline control XGBoost model')

print('Models trained successfully with XGBoost GPU params:', XGB_GPU_PARAMS)
print()

test_treated_pos = np.where(test_df['intervention_flag'].to_numpy() == 1)[0]
test_control_pos = np.where(test_df['intervention_flag'].to_numpy() == 0)[0]

pred_treated = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated, 'XGBoost Treated model')

pred_control = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control, 'XGBoost Control model')


---
## XGBOOST CV GRID SEARCH FUNCTION
---


In [ ]:
xgb_grid = [
    {'max_depth': max_depth, 'eta': eta, 'min_child_weight': min_child_weight}
    for max_depth, eta, min_child_weight in product([3, 4, 5], [0.03, 0.05, 0.10], [1, 5])
]


---
## TRAIN TREATED MODEL WITH CV GRID SEARCH
---


In [ ]:
xgb_treated_cv = fit_xgb_cv_grid(x_matrix=x_treated, y=y_treated, grid=xgb_grid, nrounds_max=500, nfold=5)
model_treated = xgb_treated_cv['model']
assert_xgb_booster_uses_cuda(model_treated, 'CV-tuned treated XGBoost model')

print('XGBoost Treated best CV AUC:', round(xgb_treated_cv['best_cv_auc'], 4))
print('XGBoost Treated best nrounds:', xgb_treated_cv['best_nrounds'])
print('XGBoost Treated best params:')
print(xgb_treated_cv['best_params'])


---
## TRAIN CONTROL MODEL WITH CV GRID SEARCH
---


In [ ]:
xgb_control_cv = fit_xgb_cv_grid(x_matrix=x_control, y=y_control, grid=xgb_grid, nrounds_max=500, nfold=5)
model_control = xgb_control_cv['model']
assert_xgb_booster_uses_cuda(model_control, 'CV-tuned control XGBoost model')

print('XGBoost Control best CV AUC:', round(xgb_control_cv['best_cv_auc'], 4))
print('XGBoost Control best nrounds:', xgb_control_cv['best_nrounds'])
print('XGBoost Control best params:')
print(xgb_control_cv['best_params'])


---
## TEST AUC FOR CV-TUNED XGBOOST MODELS
---


In [ ]:
pred_treated_cv_xgb = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_xgb, 'XGBoost Treated CV-tuned test')

pred_control_cv_xgb = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_xgb, 'XGBoost Control CV-tuned test')

print('CV-tuned XGBoost models trained successfully.')
print()


---
## CPU-ONLY GLMNET COMPARISON
---


In [ ]:
if RUN_CPU_ONLY_COMPARISON_MODELS:
    print('Training fixed GLMNET comparison models on CPU without hyperparameter tuning.')
    print()
    glmnet_shared_scaler = StandardScaler().fit(pd.concat([x_treated, x_control], axis=0))
    print('GLMNET shared scaler fit on combined treated/control training matrix.')
    enet_treated = fit_elastic_net(x_treated, y_treated, prefit_scaler=glmnet_shared_scaler)

    print('Fixed treated alpha:', enet_treated['best_alpha'])
    print('Fixed treated lambda:', enet_treated['best_lambda'])
    print('Treated CV AUC: not calculated (hyperparameter tuning disabled)')
    print()

    enet_control = fit_elastic_net(x_control, y_control, prefit_scaler=glmnet_shared_scaler)

    print('Fixed control alpha:', enet_control['best_alpha'])
    print('Fixed control lambda:', enet_control['best_lambda'])
    print('Control CV AUC: not calculated (hyperparameter tuning disabled)')
    print()
else:
    glmnet_shared_scaler = None
    enet_treated = None
    enet_control = None
    print('Skipped GLMNET comparison models because RUN_CPU_ONLY_COMPARISON_MODELS is False.')
    print()


---
## OPTIONAL TEST AUC FOR CPU-ONLY GLMNET COMPARISON
---


In [ ]:
if enet_treated is not None and enet_control is not None:
    pred_treated_cv_glmnet = enet_treated['best_model'].predict_proba(x_test.iloc[test_treated_pos])[:, 1]
    auc_treated_cv_glmnet = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_glmnet, 'GLMNET Treated CV-tuned test')

    pred_control_cv_glmnet = enet_control['best_model'].predict_proba(x_test.iloc[test_control_pos])[:, 1]
    auc_control_cv_glmnet = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_glmnet, 'GLMNET Control CV-tuned test')
else:
    pred_treated_cv_glmnet = None
    pred_control_cv_glmnet = None
    auc_treated_cv_glmnet = np.nan
    auc_control_cv_glmnet = np.nan
    print('Skipped GLMNET test AUC because CPU-only GLMNET training was skipped.')
print()


---
## 15. SCORE TEST SETS
---


In [ ]:
p_treated_xgboost = model_treated.predict(make_dmatrix(x_test))
p_control_xgboost = model_control.predict(make_dmatrix(x_test))
results_test_xgboost = build_uplift_results(test_df, p_treated_xgboost, p_control_xgboost)

# Backward-compatible alias for the primary XGBoost result.
results_test = results_test_xgboost
print_highest_benefit(results_test_xgboost, 'XGBoost')

if enet_treated is not None and enet_control is not None:
    p_treated_glmnet = enet_treated['best_model'].predict_proba(x_test)[:, 1]
    p_control_glmnet = enet_control['best_model'].predict_proba(x_test)[:, 1]
    results_test_glmnet = build_uplift_results(test_df, p_treated_glmnet, p_control_glmnet)
    print_highest_benefit(results_test_glmnet, 'GLMNET')
else:
    p_treated_glmnet = None
    p_control_glmnet = None
    results_test_glmnet = None
    print('Skipped GLMNET test scoring because CPU-only GLMNET training was skipped.')
    print()


---
## 16. DECILE SUMMARIES
---


In [ ]:
decile_summary_xgboost = summarize_uplift_deciles(results_test_xgboost)

# Backward-compatible alias for the primary XGBoost summary.
decile_summary = decile_summary_xgboost

print('XGBoost decile summary:')
display(decile_summary_xgboost)
print()

if results_test_glmnet is not None:
    decile_summary_glmnet = summarize_uplift_deciles(results_test_glmnet)
    print('GLMNET decile summary:')
    display(decile_summary_glmnet)
    print()
else:
    decile_summary_glmnet = None


---
## X-LEARNER TREATMENT EFFECT COMPARISON
---

This section trains X-learner treatment-effect models for both XGBoost and GLMNet. The goal is not to replace the T-learner output, but to provide a second uplift framework that can be compared against the T-learner benefit ranking for consistency.

### How the X-learner works in this PRISM uplift workflow

```mermaid
flowchart TD
    A["PRISM training data<br/>Member predictors X, intervention flag W, ED outcome Y"] --> B{"Split by observed intervention status"}

    B --> C["Treated members<br/>W = 1"]
    B --> D["Control members<br/>W = 0"]

    C --> E["Train treated outcome model<br/>mu1(X) = predicted ED risk if treated"]
    D --> F["Train control outcome model<br/>mu0(X) = predicted ED risk if not treated"]

    E --> G["Predict treated risk for control members<br/>mu1(X_control)"]
    F --> H["Predict control risk for treated members<br/>mu0(X_treated)"]

    H --> I["Impute treated-group benefit<br/>D1 = mu0(X_treated) - observed Y_treated"]
    G --> J["Impute control-group benefit<br/>D0 = observed Y_control - mu1(X_control)"]

    I --> K["Train effect model on treated members<br/>tau1(X) learns benefit pattern for W = 1"]
    J --> L["Train effect model on control members<br/>tau0(X) learns benefit pattern for W = 0"]

    A --> M["Train propensity model<br/>e(X) = probability member received intervention"]

    K --> N["Score test members<br/>tau1(X_test)"]
    L --> O["Score test members<br/>tau0(X_test)"]
    M --> P["Score test members<br/>e(X_test), clipped to 0.05-0.95"]

    N --> Q["Weighted X-learner benefit score<br/>benefit = e(X) * tau0(X) + (1 - e(X)) * tau1(X)"]
    O --> Q
    P --> Q

    Q --> R["Rank members into uplift deciles<br/>Decile 1 = highest predicted treatment benefit"]
    R --> S["Compare X-learner ranking with T-learner ranking<br/>correlations, top-decile overlap, decile summaries"]
```

In this notebook, a larger X-learner `benefit_score` means the intervention is predicted to reduce 90-day ED risk more for that member. The X-learner differs from the T-learner by first creating imputed individual benefit labels for treated and control members, then learning those benefit patterns directly and blending them with propensity weights.

---


In [ ]:
# X-learner implementation for treatment-effect consistency checks.
# Uses the already-trained T-learner outcome models to impute individual treatment effects,
# then trains second-stage treatment-effect models on those imputed effects.

xlearner_root_folder = ensure_output_folder(output_folder / 'X-Learner')
xlearner_xgboost_output_folder = ensure_output_folder(xlearner_root_folder / 'XGBoost')
xlearner_glmnet_output_folder = ensure_output_folder(xlearner_root_folder / 'GLMNet')


def fit_xgb_regression_model(x_matrix, y, model_label, seed=123):
    params = xgb_training_params(
        XGB_GPU_PARAMS,
        {
            'max_depth': 3,
            'eta': 0.05,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 1,
            'objective': 'reg:squarederror',
        },
        eval_metric='rmse',
        seed=seed,
    )
    model = xgb.train(
        params=params,
        dtrain=make_dmatrix(x_matrix, np.asarray(y, dtype=float)),
        num_boost_round=200,
        verbose_eval=False,
    )
    assert_xgb_booster_uses_cuda(model, model_label)
    return model


class PrefitScaledRegressionPipeline:
    def __init__(self, scaler, model):
        self.named_steps = {
            'standardscaler': scaler,
            'elasticnetcv': model,
        }

    def predict(self, x_matrix):
        return self.named_steps['elasticnetcv'].predict(
            self.named_steps['standardscaler'].transform(x_matrix)
        )


def fit_glmnet_regression_model(x_matrix, y, seed=123, prefit_scaler=None):
    y_array = np.asarray(y, dtype=float)
    if len(y_array) < 2:
        raise ValueError('Need at least two rows for GLMNET X-learner regression.')
    reg_model = ElasticNet(
        alpha=1.0,
        l1_ratio=0.5,
        max_iter=2000,
        tol=1e-3,
        random_state=seed,
    )
    if prefit_scaler is None:
        scaler = StandardScaler().fit(x_matrix)
    else:
        scaler = prefit_scaler
    reg_model.fit(scaler.transform(x_matrix), y_array)
    return PrefitScaledRegressionPipeline(scaler, reg_model)


def fit_propensity_model(x_matrix, treatment, seed=123):
    treatment_array = np.asarray(treatment, dtype=float)
    if pd.Series(treatment_array).nunique() < 2:
        raise ValueError('Need both treatment classes for propensity modeling.')
    propensity_model = LogisticRegression(
        C=1.0,
        penalty='elasticnet',
        solver='saga',
        l1_ratio=0.5,
        max_iter=2000,
        tol=1e-3,
        random_state=seed,
    )
    pipeline = make_pipeline(StandardScaler(), propensity_model)
    pipeline.fit(x_matrix, treatment_array)
    return pipeline


def clipped_propensity(propensity_model, x_matrix, lower=0.05, upper=0.95):
    propensity = propensity_model.predict_proba(x_matrix)[:, 1]
    return np.clip(propensity, lower, upper)


def build_xlearner_results(base_df, pred_treated, pred_control, cate_score, propensity_score, learner_label):
    results = base_df.copy()
    results['pred_ed_if_treated'] = pred_treated
    results['pred_ed_if_control'] = pred_control
    results['t_learner_benefit_score'] = results['pred_ed_if_control'] - results['pred_ed_if_treated']
    results['benefit_score'] = cate_score
    results['xlearner_propensity_score'] = propensity_score
    results['uplift_bad_outcome'] = -results['benefit_score']
    results['uplift_decile'] = ntile_desc(results['benefit_score'], 10).to_numpy()
    results['learner_framework'] = 'X-Learner'
    results['model'] = learner_label
    return results


def xlearner_consistency_summary(t_results, x_results, model_label):
    merged = pd.DataFrame(
        {
            't_learner_benefit_score': t_results['benefit_score'].to_numpy(),
            'x_learner_benefit_score': x_results['benefit_score'].to_numpy(),
            't_learner_decile': t_results['uplift_decile'].to_numpy(),
            'x_learner_decile': x_results['uplift_decile'].to_numpy(),
        },
        index=t_results.index,
    )
    t_top = set(merged.index[merged['t_learner_decile'] == 1])
    x_top = set(merged.index[merged['x_learner_decile'] == 1])
    top_overlap = len(t_top & x_top) / len(t_top) if t_top else np.nan
    return pd.DataFrame(
        [
            {
                'model': model_label,
                'pearson_benefit_score_corr': merged['t_learner_benefit_score'].corr(merged['x_learner_benefit_score'], method='pearson'),
                'spearman_benefit_score_corr': merged['t_learner_benefit_score'].corr(merged['x_learner_benefit_score'], method='spearman'),
                'top_decile_overlap_pct': top_overlap,
                't_learner_mean_benefit_score': merged['t_learner_benefit_score'].mean(),
                'x_learner_mean_benefit_score': merged['x_learner_benefit_score'].mean(),
            }
        ]
    )


x_train_all = pd.concat([x_treated, x_control], axis=0)
t_train_all = pd.concat(
    [
        pd.Series(np.ones(len(x_treated)), index=x_treated.index),
        pd.Series(np.zeros(len(x_control)), index=x_control.index),
    ],
    axis=0,
)
propensity_model = fit_propensity_model(x_train_all, t_train_all)
propensity_train_all = clipped_propensity(propensity_model, x_train_all)
propensity_test = clipped_propensity(propensity_model, x_test)

shared_propensity_scores = pd.concat(
    [
        pd.DataFrame({
            'member_id': pd.concat([train_treated['member_id'], train_control['member_id']], axis=0).to_numpy(),
            'split': 'train',
            'propensity_score': propensity_train_all,
            'propensity_model': 'GLMNet elastic-net logistic',
            'seed': 123,
        }),
        pd.DataFrame({
            'member_id': test_df['member_id'].to_numpy(),
            'split': 'test',
            'propensity_score': propensity_test,
            'propensity_model': 'GLMNet elastic-net logistic',
            'seed': 123,
        }),
    ],
    ignore_index=True,
).sort_values('member_id').reset_index(drop=True)
shared_propensity_path = xlearner_root_folder / 'shared_propensity_scores.csv'
shared_propensity_scores.to_csv(shared_propensity_path, index=False)
print('Shared member-level propensity scores written to:', shared_propensity_path)

# XGBoost X-learner: impute benefit effects on the same scale as the T-learner.
# Positive benefit means lower ED risk under treatment: control outcome/risk - treated outcome/risk.
xgb_mu0_on_treated = model_control.predict(make_dmatrix(x_treated))
xgb_mu1_on_control = model_treated.predict(make_dmatrix(x_control))
xgb_effect_treated = xgb_mu0_on_treated - np.asarray(y_treated, dtype=float)
xgb_effect_control = np.asarray(y_control, dtype=float) - xgb_mu1_on_control

xgb_tau_treated_model = fit_xgb_regression_model(
    x_treated,
    xgb_effect_treated,
    'XGBoost X-learner treated-effect model',
)
xgb_tau_control_model = fit_xgb_regression_model(
    x_control,
    xgb_effect_control,
    'XGBoost X-learner control-effect model',
)

xgb_tau_treated_test = xgb_tau_treated_model.predict(make_dmatrix(x_test))
xgb_tau_control_test = xgb_tau_control_model.predict(make_dmatrix(x_test))
xgb_xlearner_benefit = propensity_test * xgb_tau_control_test + (1 - propensity_test) * xgb_tau_treated_test

results_test_xlearner_xgboost = build_xlearner_results(
    test_df,
    p_treated_xgboost,
    p_control_xgboost,
    xgb_xlearner_benefit,
    propensity_test,
    'XGBoost',
)
decile_summary_xlearner_xgboost = summarize_uplift_deciles(results_test_xlearner_xgboost)
print('XGBoost X-learner decile summary:')
display(decile_summary_xlearner_xgboost)
print_highest_benefit(results_test_xlearner_xgboost, 'XGBoost X-Learner')

results_test_xlearner_xgboost.to_csv(xlearner_xgboost_output_folder / 'xlearner_scored_test_output.csv', index=False)
decile_summary_xlearner_xgboost.to_csv(xlearner_xgboost_output_folder / 'xlearner_decile_summary.csv', index=False)

# GLMNET X-learner: impute benefit effects on the same control-minus-treated scale,
# then fit elastic-net second-stage treatment-effect regressions.
if enet_treated is not None and enet_control is not None:
    glmnet_mu0_on_treated = enet_control['best_model'].predict_proba(x_treated)[:, 1]
    glmnet_mu1_on_control = enet_treated['best_model'].predict_proba(x_control)[:, 1]
    glmnet_effect_treated = glmnet_mu0_on_treated - np.asarray(y_treated, dtype=float)
    glmnet_effect_control = np.asarray(y_control, dtype=float) - glmnet_mu1_on_control

    xlearner_effect_shared_scaler = StandardScaler().fit(pd.concat([x_treated, x_control], axis=0))
    print('GLMNET X-learner shared effect scaler fit on combined treated/control training matrix.')
    glmnet_tau_treated_model = fit_glmnet_regression_model(
        x_treated,
        glmnet_effect_treated,
        prefit_scaler=xlearner_effect_shared_scaler,
    )
    glmnet_tau_control_model = fit_glmnet_regression_model(
        x_control,
        glmnet_effect_control,
        prefit_scaler=xlearner_effect_shared_scaler,
    )

    glmnet_tau_treated_test = glmnet_tau_treated_model.predict(x_test)
    glmnet_tau_control_test = glmnet_tau_control_model.predict(x_test)
    glmnet_xlearner_benefit = propensity_test * glmnet_tau_control_test + (1 - propensity_test) * glmnet_tau_treated_test

    results_test_xlearner_glmnet = build_xlearner_results(
        test_df,
        p_treated_glmnet,
        p_control_glmnet,
        glmnet_xlearner_benefit,
        propensity_test,
        'GLMNET',
    )
    decile_summary_xlearner_glmnet = summarize_uplift_deciles(results_test_xlearner_glmnet)
    print('GLMNET X-learner decile summary:')
    display(decile_summary_xlearner_glmnet)
    print_highest_benefit(results_test_xlearner_glmnet, 'GLMNET X-Learner')

    results_test_xlearner_glmnet.to_csv(xlearner_glmnet_output_folder / 'xlearner_scored_test_output.csv', index=False)
    decile_summary_xlearner_glmnet.to_csv(xlearner_glmnet_output_folder / 'xlearner_decile_summary.csv', index=False)
else:
    xlearner_effect_shared_scaler = None
    glmnet_tau_treated_model = None
    glmnet_tau_control_model = None
    results_test_xlearner_glmnet = None
    decile_summary_xlearner_glmnet = None
    print('Skipped GLMNET X-learner because GLMNET T-learner outcome models were not available.')

consistency_frames = [
    xlearner_consistency_summary(results_test_xgboost, results_test_xlearner_xgboost, 'XGBoost'),
]
if results_test_glmnet is not None and results_test_xlearner_glmnet is not None:
    consistency_frames.append(
        xlearner_consistency_summary(results_test_glmnet, results_test_xlearner_glmnet, 'GLMNET')
    )

xlearner_consistency = pd.concat(consistency_frames, ignore_index=True)
xlearner_consistency.to_csv(xlearner_root_folder / 'xlearner_vs_tlearner_consistency_summary.csv', index=False)
print('X-learner versus T-learner consistency summary:')
display(xlearner_consistency)
print('X-learner outputs saved to:', xlearner_root_folder)


def save_paired_glmnet_benefit_decile_charts(t_decile_df, x_decile_df):
    max_benefit = max(
        t_decile_df['avg_benefit_score'].max(),
        x_decile_df['avg_benefit_score'].max(),
    )
    shared_y_max = np.ceil((max_benefit * 1.08) / 0.005) * 0.005

    chart_specs = [
        (
            t_decile_df,
            glmnet_output_folder / 'dashboard_avg_benefit_by_decile.png',
            'GLMNet T-Learner: Average Predicted Benefit by Decile',
        ),
        (
            x_decile_df,
            xlearner_glmnet_output_folder / 'dashboard_avg_benefit_by_decile.png',
            'GLMNet X-Learner: Average Predicted Benefit by Decile',
        ),
    ]

    for decile_df, chart_path, title in chart_specs:
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        ax.bar(
            decile_df['uplift_decile'].astype(str),
            decile_df['avg_benefit_score'],
            color='#4f76b5',
        )
        ax.set_title(title)
        ax.set_xlabel('Uplift Decile: 1 = Highest Predicted Benefit')
        ax.set_ylabel('Average Predicted Benefit')
        ax.set_ylim(0, shared_y_max)
        ax.grid(axis='y', alpha=0.35)
        ax.set_axisbelow(True)
        fig.tight_layout()
        fig.savefig(chart_path, dpi=200, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved paired GLMNet benefit decile charts with shared y-axis max {shared_y_max:.3f}.')


if decile_summary_glmnet is not None and decile_summary_xlearner_glmnet is not None:
    save_paired_glmnet_benefit_decile_charts(
        decile_summary_glmnet,
        decile_summary_xlearner_glmnet,
    )
else:
    print('Skipped paired GLMNet benefit decile charts because one or more decile summaries were unavailable.')


---
## XGBOOST BENEFIT BY OBSERVED TREATMENT AND ACTUAL OUTCOME
---

This diagnostic compares predicted benefit across the four factual test-set groups:
treated/outcome 1, treated/outcome 0, untreated/outcome 1, and untreated/outcome 0.
It is generated separately for the XGBoost T-Learner and XGBoost X-Learner.

Benefit is defined as the predicted 90-day ED risk if untreated minus the predicted
risk if treated, so negative benefit indicates adverse expected benefit (a "sleeping
dog"). The grouped results are descriptive. Because each member has only one observed
treatment and one observed outcome, these tables do not reveal the member's unobserved
counterfactual outcome and cannot establish that treatment caused an observed event.


In [ ]:
def summarize_benefit_by_observed_group(results, learner_label):
    required_columns = {'intervention_flag', 'outcome_ed_90d', 'benefit_score'}
    missing_columns = required_columns.difference(results.columns)
    if missing_columns:
        raise ValueError(
            f'{learner_label} results are missing required columns: {sorted(missing_columns)}'
        )

    diagnostic = results.loc[:, sorted(required_columns)].copy()
    diagnostic['intervention_flag'] = pd.to_numeric(
        diagnostic['intervention_flag'], errors='raise'
    ).astype(int)
    diagnostic['outcome_ed_90d'] = pd.to_numeric(
        diagnostic['outcome_ed_90d'], errors='raise'
    ).astype(int)

    if not diagnostic['intervention_flag'].isin([0, 1]).all():
        raise ValueError('intervention_flag must be binary for this diagnostic.')
    if not diagnostic['outcome_ed_90d'].isin([0, 1]).all():
        raise ValueError('outcome_ed_90d must be binary for this diagnostic.')

    summary = (
        diagnostic.groupby(
            ['intervention_flag', 'outcome_ed_90d'],
            observed=True,
        )
        .agg(
            members=('benefit_score', 'size'),
            mean_benefit=('benefit_score', 'mean'),
            benefit_p25=('benefit_score', lambda values: values.quantile(0.25)),
            median_benefit=('benefit_score', 'median'),
            benefit_p75=('benefit_score', lambda values: values.quantile(0.75)),
            negative_benefit_members=(
                'benefit_score',
                lambda values: int((values < 0).sum()),
            ),
            negative_benefit_pct=(
                'benefit_score',
                lambda values: float((values < 0).mean()),
            ),
        )
        .reindex([(1, 1), (1, 0), (0, 1), (0, 0)])
        .reset_index()
    )

    summary.insert(0, 'learner', learner_label)
    summary.insert(
        2,
        'observed_treatment_group',
        summary['intervention_flag'].map({1: 'Treated', 0: 'Untreated'}),
    )
    summary.insert(
        4,
        'actual_outcome_group',
        summary['outcome_ed_90d'].map(
            {1: 'Actual outcome = 1', 0: 'Actual outcome = 0'}
        ),
    )
    return summary


tlearner_xgboost_treatment_outcome_benefit_summary = (
    summarize_benefit_by_observed_group(
        results_test_xgboost,
        'XGBoost T-Learner',
    )
)
xlearner_xgboost_treatment_outcome_benefit_summary = (
    summarize_benefit_by_observed_group(
        results_test_xlearner_xgboost,
        'XGBoost X-Learner',
    )
)

tlearner_xgboost_treatment_outcome_benefit_summary.to_csv(
    xgboost_output_folder / 'treatment_outcome_benefit_summary.csv',
    index=False,
)
xlearner_xgboost_treatment_outcome_benefit_summary.to_csv(
    xlearner_xgboost_output_folder / 'treatment_outcome_benefit_summary.csv',
    index=False,
)

def format_benefit_summary_for_display(summary):
    formatted = summary.copy()
    for column in [
        'mean_benefit',
        'benefit_p25',
        'median_benefit',
        'benefit_p75',
    ]:
        formatted[column] = formatted[column].map(lambda value: f'{value:.2%}')
    if 'negative_benefit_pct' in formatted.columns:
        formatted['negative_benefit_pct'] = formatted['negative_benefit_pct'].map(
            lambda value: f'{value:.1%}'
        )
    return formatted

print('XGBoost T-Learner: benefit by observed treatment and actual outcome (test set)')
display(
    format_benefit_summary_for_display(
        tlearner_xgboost_treatment_outcome_benefit_summary
    )
)

print('XGBoost X-Learner: benefit by observed treatment and actual outcome (test set)')
display(
    format_benefit_summary_for_display(
        xlearner_xgboost_treatment_outcome_benefit_summary
    )
)


---
## OVERLAP-WEIGHTED XGBOOST T-LEARNER
---

This sensitivity model uses the same pre-treatment elastic-net logistic propensity
model already fitted for the X-Learner. Treated observations receive weight
`1 - propensity`, and untreated observations receive weight `propensity`. Weights are
normalized to mean 1 within each treatment arm so XGBoost's regularization scale remains
comparable to the unweighted models.

To keep this focused and computationally efficient, the weighted models reuse the
corresponding unweighted T-Learner's selected hyperparameters and boosting rounds.
The outputs below are limited to the requested four-group benefit summary and the same
benefit-by-decile summary/chart used for the unweighted XGBoost T-Learner.


In [ ]:
overlap_weighted_xgboost_output_folder = ensure_output_folder(
    tlearner_root_folder / 'XGBoost_Overlap_Weighted'
)


def normalized_overlap_weights(propensity_scores, observed_treatment):
    propensity_array = np.asarray(propensity_scores, dtype=float)
    if observed_treatment == 1:
        raw_weights = 1.0 - propensity_array
    elif observed_treatment == 0:
        raw_weights = propensity_array
    else:
        raise ValueError('observed_treatment must be 0 or 1.')

    if not np.isfinite(raw_weights).all() or raw_weights.mean() <= 0:
        raise ValueError('Overlap weights must be finite with a positive mean.')
    return raw_weights / raw_weights.mean()


propensity_treated_train = clipped_propensity(propensity_model, x_treated)
propensity_control_train = clipped_propensity(propensity_model, x_control)

overlap_weights_treated = normalized_overlap_weights(
    propensity_treated_train,
    observed_treatment=1,
)
overlap_weights_control = normalized_overlap_weights(
    propensity_control_train,
    observed_treatment=0,
)

overlap_treated_dmatrix = xgb.DMatrix(
    x_treated,
    label=np.asarray(y_treated, dtype=float),
    weight=overlap_weights_treated,
    feature_names=list(x_treated.columns),
)
overlap_control_dmatrix = xgb.DMatrix(
    x_control,
    label=np.asarray(y_control, dtype=float),
    weight=overlap_weights_control,
    feature_names=list(x_control.columns),
)

overlap_weighted_model_treated = xgb.train(
    params=xgb_treated_cv['best_params'],
    dtrain=overlap_treated_dmatrix,
    num_boost_round=xgb_treated_cv['best_nrounds'],
    verbose_eval=False,
)
overlap_weighted_model_control = xgb.train(
    params=xgb_control_cv['best_params'],
    dtrain=overlap_control_dmatrix,
    num_boost_round=xgb_control_cv['best_nrounds'],
    verbose_eval=False,
)

assert_xgb_booster_uses_cuda(
    overlap_weighted_model_treated,
    'Overlap-weighted treated XGBoost model',
)
assert_xgb_booster_uses_cuda(
    overlap_weighted_model_control,
    'Overlap-weighted control XGBoost model',
)

overlap_pred_treated_test = overlap_weighted_model_treated.predict(
    make_dmatrix(x_test)
)
overlap_pred_control_test = overlap_weighted_model_control.predict(
    make_dmatrix(x_test)
)
results_test_overlap_weighted_xgboost = build_uplift_results(
    test_df,
    overlap_pred_treated_test,
    overlap_pred_control_test,
)
results_test_overlap_weighted_xgboost['propensity_score'] = propensity_test
results_test_overlap_weighted_xgboost['learner_framework'] = (
    'Overlap-Weighted T-Learner'
)
results_test_overlap_weighted_xgboost['model'] = 'XGBoost'

overlap_weighted_decile_summary_xgboost = summarize_uplift_deciles(
    results_test_overlap_weighted_xgboost
)
overlap_weighted_treatment_outcome_benefit_summary = (
    summarize_benefit_by_observed_group(
        results_test_overlap_weighted_xgboost,
        'Overlap-Weighted XGBoost T-Learner',
    )
)
overlap_weighted_treatment_outcome_benefit_summary = (
    overlap_weighted_treatment_outcome_benefit_summary[
        [
            'learner',
            'intervention_flag',
            'observed_treatment_group',
            'outcome_ed_90d',
            'actual_outcome_group',
            'members',
            'mean_benefit',
            'benefit_p25',
            'median_benefit',
            'benefit_p75',
        ]
    ].copy()
)

results_test_overlap_weighted_xgboost.to_csv(
    overlap_weighted_xgboost_output_folder
    / 'overlap_weighted_scored_test_output.csv',
    index=False,
)
overlap_weighted_decile_summary_xgboost.to_csv(
    overlap_weighted_xgboost_output_folder / 'uplift_decile_summary.csv',
    index=False,
)
overlap_weighted_treatment_outcome_benefit_summary.to_csv(
    overlap_weighted_xgboost_output_folder
    / 'treatment_outcome_benefit_summary.csv',
    index=False,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    overlap_weighted_decile_summary_xgboost['uplift_decile'].astype(str),
    overlap_weighted_decile_summary_xgboost['avg_benefit_score'],
)
ax.set_title(
    'Overlap-Weighted XGBoost T-Learner: Average Predicted Intervention Benefit by Uplift Decile'
)
ax.set_xlabel('Uplift Decile: 1 = Highest Predicted Benefit')
ax.set_ylabel('Average Benefit Score')
fig.tight_layout()
fig.savefig(
    overlap_weighted_xgboost_output_folder
    / 'dashboard_avg_benefit_by_decile.png',
    dpi=150,
)
plt.close(fig)

print(
    'Overlap-weighted XGBoost T-Learner: benefit by observed treatment '
    'and actual outcome (test set)'
)
display(
    format_benefit_summary_for_display(
        overlap_weighted_treatment_outcome_benefit_summary
    )
)

print('Overlap-weighted XGBoost T-Learner benefit-by-decile summary:')
display(overlap_weighted_decile_summary_xgboost)
print('Outputs saved to:', overlap_weighted_xgboost_output_folder)


---
## WRITE-UP PROBABILITY CALIBRATION AND BRIER SCORES
---


In [ ]:
def brier_scores_for_results(results, model_label):
    if results is None:
        return None

    rows = []
    for group_label, treatment_value, pred_col in [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        if subgroup.empty:
            rows.append({
                'model': model_label,
                'group': group_label,
                'n': 0,
                'observed_ed_rate': np.nan,
                'avg_predicted_ed_rate': np.nan,
                'brier_score': np.nan,
            })
            continue

        rows.append({
            'model': model_label,
            'group': group_label,
            'n': len(subgroup),
            'observed_ed_rate': subgroup['outcome_ed_90d'].mean(),
            'avg_predicted_ed_rate': subgroup[pred_col].mean(),
            'brier_score': brier_score_loss(subgroup['outcome_ed_90d'], subgroup[pred_col]),
        })

    return pd.DataFrame(rows)


def calibration_tables_for_results(results, model_label):
    if results is None:
        return None, None

    calibration_parts = []
    for group_label, treatment_value, pred_col in [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        if subgroup.empty:
            continue

        subgroup['pred_risk_decile'] = ntile_desc(subgroup[pred_col], 10).to_numpy()
        by_decile = (
            subgroup
            .groupby('pred_risk_decile', as_index=False)
            .agg(
                n=('outcome_ed_90d', 'size'),
                avg_predicted_ed_rate=(pred_col, 'mean'),
                observed_ed_rate=('outcome_ed_90d', 'mean'),
            )
            .sort_values('pred_risk_decile')
        )
        by_decile.insert(0, 'group', group_label)
        by_decile.insert(0, 'model', model_label)
        by_decile['calibration_error'] = by_decile['observed_ed_rate'] - by_decile['avg_predicted_ed_rate']
        by_decile['abs_calibration_error'] = by_decile['calibration_error'].abs()
        calibration_parts.append(by_decile)

    if not calibration_parts:
        return None, None

    calibration_by_decile = pd.concat(calibration_parts, ignore_index=True)
    calibration_summary_rows = []
    for (model_name, group_label), frame in calibration_by_decile.groupby(['model', 'group']):
        calibration_summary_rows.append(
            {
                'model': model_name,
                'group': group_label,
                'n': frame['n'].sum(),
                'mean_abs_calibration_error': frame['abs_calibration_error'].mean(),
                'weighted_mean_abs_calibration_error': np.average(
                    frame['abs_calibration_error'],
                    weights=frame['n'],
                ),
                'max_abs_calibration_error': frame['abs_calibration_error'].max(),
            }
        )
    calibration_summary = pd.DataFrame(calibration_summary_rows)
    return calibration_by_decile, calibration_summary


def save_calibration_plot(calibration_by_decile, folder, model_label):
    if calibration_by_decile is None or calibration_by_decile.empty:
        return

    groups = [group for group in ['Control', 'Treated'] if group in set(calibration_by_decile['group'])]
    fig, axes = plt.subplots(1, len(groups), figsize=(7 * len(groups), 5), sharey=True)
    if len(groups) == 1:
        axes = [axes]

    max_rate = max(
        calibration_by_decile['avg_predicted_ed_rate'].max(),
        calibration_by_decile['observed_ed_rate'].max(),
        0.01,
    )

    for ax, group_label in zip(axes, groups):
        group_df = calibration_by_decile[calibration_by_decile['group'] == group_label].sort_values('pred_risk_decile')
        x_values = np.arange(len(group_df))
        bar_width = 0.38

        ax.bar(
            x_values - bar_width / 2,
            group_df['avg_predicted_ed_rate'],
            width=bar_width,
            label='Avg predicted ED rate',
            color='#4C78A8',
        )
        ax.bar(
            x_values + bar_width / 2,
            group_df['observed_ed_rate'],
            width=bar_width,
            label='Observed ED rate',
            color='#F58518',
        )

        for x_pos, n_value in zip(x_values, group_df['n']):
            ax.text(x_pos, max_rate * 1.03, f'n={int(n_value)}', ha='center', va='bottom', fontsize=8, rotation=90)

        ax.set_title(f'{model_label}: {group_label} Calibration')
        ax.set_xlabel('Predicted Risk Decile: 1 = Highest Predicted Risk')
        ax.set_xticks(x_values)
        ax.set_xticklabels(group_df['pred_risk_decile'].astype(str))
        ax.set_ylim(0, max_rate * 1.22)
        ax.grid(axis='y', alpha=0.25)

    axes[0].set_ylabel('ED Rate')
    axes[-1].legend(loc='upper right')
    fig.suptitle(f'{model_label}: Predicted vs Observed ED Rate by Risk Decile', y=1.03)
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_calibration_plot.png', dpi=150, bbox_inches='tight')
    plt.close(fig)


def save_probability_evaluation_outputs(results, folder, model_label):
    brier_df = brier_scores_for_results(results, model_label)
    calibration_by_decile, calibration_summary = calibration_tables_for_results(results, model_label)

    if brier_df is not None:
        brier_df.to_csv(folder / 'model_brier_scores.csv', index=False)
        print(f'{model_label} Brier scores:')
        display(brier_df)

    if calibration_by_decile is not None:
        calibration_by_decile.to_csv(folder / 'calibration_by_decile.csv', index=False)
        calibration_summary.to_csv(folder / 'calibration_summary.csv', index=False)
        save_calibration_plot(calibration_by_decile, folder, model_label)
        print(f'{model_label} calibration summary:')
        display(calibration_summary)
        print()

    return brier_df, calibration_by_decile, calibration_summary


brier_xgboost, calibration_by_decile_xgboost, calibration_summary_xgboost = save_probability_evaluation_outputs(
    results_test_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None:
    brier_glmnet, calibration_by_decile_glmnet, calibration_summary_glmnet = save_probability_evaluation_outputs(
        results_test_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    brier_glmnet = None
    calibration_by_decile_glmnet = None
    calibration_summary_glmnet = None
    print('Skipped GLMNET Brier/calibration outputs because GLMNET scoring was not available.')


---
## WRITE-UP FACTUAL OUTCOME MODEL DIAGNOSTICS
---


In [ ]:
def factual_event_count_summary(train_df, test_df):
    rows = []
    for split_name, frame in [('Train', train_df), ('Test', test_df)]:
        for group_label, treatment_value in [('Treated', 1), ('Control', 0)]:
            subgroup = frame[frame['intervention_flag'] == treatment_value]
            n = len(subgroup)
            events = int(subgroup['outcome_ed_90d'].sum()) if n else 0
            rows.append(
                {
                    'split': split_name,
                    'group': group_label,
                    'n': n,
                    'positive_ed_events': events,
                    'negative_ed_events': n - events,
                    'event_rate': events / n if n else np.nan,
                }
            )
    return pd.DataFrame(rows)


def factual_prediction_diagnostics(results, model_label):
    separation_rows = []
    threshold_rows = []
    range_rows = []
    specs = [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]

    for group_label, treatment_value, pred_col in specs:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        n = len(subgroup)
        events = int(subgroup['outcome_ed_90d'].sum()) if n else 0
        non_events = n - events
        positives = subgroup[subgroup['outcome_ed_90d'] == 1]
        negatives = subgroup[subgroup['outcome_ed_90d'] == 0]
        auc_value = roc_auc_score(subgroup['outcome_ed_90d'], subgroup[pred_col]) if subgroup['outcome_ed_90d'].nunique() == 2 else np.nan

        separation_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'n': n,
                'positive_ed_events': events,
                'negative_ed_events': non_events,
                'event_rate': events / n if n else np.nan,
                'auc': auc_value,
                'avg_pred_actual_positive': positives[pred_col].mean() if len(positives) else np.nan,
                'avg_pred_actual_negative': negatives[pred_col].mean() if len(negatives) else np.nan,
                'avg_pred_positive_minus_negative': (
                    positives[pred_col].mean() - negatives[pred_col].mean()
                    if len(positives) and len(negatives)
                    else np.nan
                ),
                'median_pred_actual_positive': positives[pred_col].median() if len(positives) else np.nan,
                'median_pred_actual_negative': negatives[pred_col].median() if len(negatives) else np.nan,
                'median_pred_positive_minus_negative': (
                    positives[pred_col].median() - negatives[pred_col].median()
                    if len(positives) and len(negatives)
                    else np.nan
                ),
            }
        )

        threshold = events / n if n else np.nan
        predicted_positive = subgroup[pred_col] >= threshold if n and not pd.isna(threshold) else pd.Series(False, index=subgroup.index)
        actual_positive = subgroup['outcome_ed_90d'] == 1
        true_positive = int((predicted_positive & actual_positive).sum())
        false_positive = int((predicted_positive & ~actual_positive).sum())
        true_negative = int((~predicted_positive & ~actual_positive).sum())
        false_negative = int((~predicted_positive & actual_positive).sum())
        predicted_positive_n = true_positive + false_positive
        precision = true_positive / predicted_positive_n if predicted_positive_n else np.nan
        recall = true_positive / events if events else np.nan

        threshold_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'threshold_rule': 'factual_group_test_event_rate',
                'threshold': threshold,
                'n': n,
                'actual_positive_n': events,
                'actual_negative_n': non_events,
                'predicted_positive_n': predicted_positive_n,
                'predicted_negative_n': true_negative + false_negative,
                'true_positive': true_positive,
                'false_positive': false_positive,
                'true_negative': true_negative,
                'false_negative': false_negative,
                'precision': precision,
                'recall': recall,
            }
        )

        predictions = subgroup[pred_col].dropna()
        range_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'n': n,
                'min_pred': predictions.min() if len(predictions) else np.nan,
                'p10_pred': predictions.quantile(0.10) if len(predictions) else np.nan,
                'median_pred': predictions.median() if len(predictions) else np.nan,
                'mean_pred': predictions.mean() if len(predictions) else np.nan,
                'p90_pred': predictions.quantile(0.90) if len(predictions) else np.nan,
                'max_pred': predictions.max() if len(predictions) else np.nan,
            }
        )

    return pd.DataFrame(separation_rows), pd.DataFrame(threshold_rows), pd.DataFrame(range_rows)


event_count_summary = factual_event_count_summary(train_df, test_df)
event_count_summary_path = output_folder / 'factual_event_count_summary.csv'
event_count_summary.to_csv(event_count_summary_path, index=False)

diagnostic_frames = []
threshold_frames = []
range_frames = []

sep_xgboost, threshold_xgboost, range_xgboost = factual_prediction_diagnostics(results_test_xgboost, 'XGBoost')
diagnostic_frames.append(sep_xgboost)
threshold_frames.append(threshold_xgboost)
range_frames.append(range_xgboost)

if results_test_glmnet is not None:
    sep_glmnet, threshold_glmnet, range_glmnet = factual_prediction_diagnostics(results_test_glmnet, 'GLMNET')
    diagnostic_frames.append(sep_glmnet)
    threshold_frames.append(threshold_glmnet)
    range_frames.append(range_glmnet)

factual_prediction_separation = pd.concat(diagnostic_frames, ignore_index=True)
factual_event_rate_threshold_classification = pd.concat(threshold_frames, ignore_index=True)
factual_prediction_ranges = pd.concat(range_frames, ignore_index=True)

factual_prediction_separation_path = output_folder / 'factual_prediction_separation.csv'
factual_prediction_ranges_path = output_folder / 'factual_prediction_ranges.csv'

factual_prediction_separation.to_csv(factual_prediction_separation_path, index=False)
factual_prediction_ranges.to_csv(factual_prediction_ranges_path, index=False)

print('Factual train/test event counts:')
display(event_count_summary)
print('Factual positive-vs-negative prediction separation:')
display(factual_prediction_separation)
print('Event-rate threshold classification:')
display(factual_event_rate_threshold_classification)
print('Prediction ranges:')
display(factual_prediction_ranges)
print('Factual diagnostic outputs written to:', output_folder)


---
## WRITE-UP OBSERVED TREATED-CONTROL GAP BY UPLIFT DECILE
---


In [ ]:
def wilson_score_interval(events, n, confidence=0.95):
    """Wilson score confidence interval for one binomial proportion."""
    if n == 0:
        return np.nan, np.nan

    z_lookup = {0.90: 1.644854, 0.95: 1.959964, 0.99: 2.575829}
    z_value = z_lookup.get(confidence, 1.959964)
    p_hat = events / n
    denominator = 1 + (z_value ** 2 / n)
    center = (p_hat + (z_value ** 2 / (2 * n))) / denominator
    half_width = (
        z_value
        * np.sqrt((p_hat * (1 - p_hat) / n) + (z_value ** 2 / (4 * n ** 2)))
        / denominator
    )
    return max(0, center - half_width), min(1, center + half_width)


def difference_in_proportions_ci(control_events, control_n, treated_events, treated_n, confidence=0.95):
    """Newcombe/Wilson CI for control observed ED rate minus treated observed ED rate."""
    if control_n == 0 or treated_n == 0:
        return np.nan, np.nan, np.nan

    control_rate = control_events / control_n
    treated_rate = treated_events / treated_n
    gap = control_rate - treated_rate
    control_lower, control_upper = wilson_score_interval(control_events, control_n, confidence)
    treated_lower, treated_upper = wilson_score_interval(treated_events, treated_n, confidence)
    gap_lower = gap - np.sqrt(
        (control_rate - control_lower) ** 2
        + (treated_upper - treated_rate) ** 2
    )
    gap_upper = gap + np.sqrt(
        (control_upper - control_rate) ** 2
        + (treated_rate - treated_lower) ** 2
    )
    gap_lower = max(-1.0, gap_lower)
    gap_upper = min(1.0, gap_upper)
    return gap, gap_lower, gap_upper


def observed_gap_by_decile(results, model_label):
    if results is None:
        return None

    rows = []
    for decile, decile_df in results.groupby('uplift_decile'):
        treated = decile_df[decile_df['intervention_flag'] == 1]
        control = decile_df[decile_df['intervention_flag'] == 0]
        treated_events = int(treated['outcome_ed_90d'].sum()) if not treated.empty else 0
        control_events = int(control['outcome_ed_90d'].sum()) if not control.empty else 0
        treated_rate = treated_events / len(treated) if len(treated) else np.nan
        control_rate = control_events / len(control) if len(control) else np.nan
        gap, gap_ci_lower, gap_ci_upper = difference_in_proportions_ci(
            control_events,
            len(control),
            treated_events,
            len(treated),
        )
        rows.append({
            'model': model_label,
            'uplift_decile': decile,
            'n': len(decile_df),
            'treated_n': len(treated),
            'control_n': len(control),
            'treated_events': treated_events,
            'control_events': control_events,
            'observed_ed_rate': decile_df['outcome_ed_90d'].mean(),
            'treated_observed_ed_rate': treated_rate,
            'control_observed_ed_rate': control_rate,
            'observed_control_minus_treated_gap': gap,
            'observed_gap_ci_lower_95': gap_ci_lower,
            'observed_gap_ci_upper_95': gap_ci_upper,
            'avg_predicted_benefit': decile_df['benefit_score'].mean(),
            'predicted_benefit_within_observed_gap_ci_95': (
                bool(gap_ci_lower <= decile_df['benefit_score'].mean() <= gap_ci_upper)
                if pd.notna(gap_ci_lower) else np.nan
            ),
            'predicted_minus_observed_gap': decile_df['benefit_score'].mean() - gap if pd.notna(gap) else np.nan,
            'abs_predicted_minus_observed_gap': abs(decile_df['benefit_score'].mean() - gap) if pd.notna(gap) else np.nan,
            'avg_pred_ed_if_treated': decile_df['pred_ed_if_treated'].mean(),
            'avg_pred_ed_if_control': decile_df['pred_ed_if_control'].mean(),
            'treated_pct': decile_df['intervention_flag'].mean(),
            'ci_method': 'newcombe_wilson_difference_in_proportions',
        })

    return pd.DataFrame(rows).sort_values('uplift_decile').reset_index(drop=True)


def uplift_curve_by_decile(results, model_label):
    if results is None:
        return None

    rows = []
    sorted_deciles = sorted(results['uplift_decile'].dropna().unique())
    total_n = len(results)
    for max_decile in sorted_deciles:
        curve_df = results[results['uplift_decile'] <= max_decile].copy()
        treated = curve_df[curve_df['intervention_flag'] == 1]
        control = curve_df[curve_df['intervention_flag'] == 0]
        treated_events = int(treated['outcome_ed_90d'].sum()) if not treated.empty else 0
        control_events = int(control['outcome_ed_90d'].sum()) if not control.empty else 0
        gap, gap_ci_lower, gap_ci_upper = difference_in_proportions_ci(
            control_events,
            len(control),
            treated_events,
            len(treated),
        )
        rows.append({
            'model': model_label,
            'through_uplift_decile': max_decile,
            'population_fraction_targeted': len(curve_df) / total_n if total_n else np.nan,
            'n': len(curve_df),
            'treated_n': len(treated),
            'control_n': len(control),
            'treated_events': treated_events,
            'control_events': control_events,
            'avg_predicted_benefit': curve_df['benefit_score'].mean(),
            'observed_control_minus_treated_gap': gap,
            'observed_gap_ci_lower_95': gap_ci_lower,
            'observed_gap_ci_upper_95': gap_ci_upper,
            'predicted_benefit_within_observed_gap_ci_95': (
                bool(gap_ci_lower <= curve_df['benefit_score'].mean() <= gap_ci_upper)
                if pd.notna(gap_ci_lower) else np.nan
            ),
        })

    curve_df = pd.DataFrame(rows).sort_values('through_uplift_decile').reset_index(drop=True)
    return curve_df


def uplift_curve_summary(curve_df, model_label):
    if curve_df is None or curve_df.empty:
        return None

    valid_observed = curve_df[['population_fraction_targeted', 'observed_control_minus_treated_gap']].dropna()
    valid_predicted = curve_df[['population_fraction_targeted', 'avg_predicted_benefit']].dropna()
    observed_area = np.trapezoid(
        valid_observed['observed_control_minus_treated_gap'],
        valid_observed['population_fraction_targeted'],
    ) if len(valid_observed) >= 2 else np.nan
    predicted_area = np.trapezoid(
        valid_predicted['avg_predicted_benefit'],
        valid_predicted['population_fraction_targeted'],
    ) if len(valid_predicted) >= 2 else np.nan

    return pd.DataFrame([
        {
            'model': model_label,
            'observed_gap_area_under_curve': observed_area,
            'predicted_benefit_area_under_curve': predicted_area,
            'final_cumulative_observed_gap': curve_df['observed_control_minus_treated_gap'].iloc[-1],
            'final_cumulative_avg_predicted_benefit': curve_df['avg_predicted_benefit'].iloc[-1],
        }
    ])


def save_observed_gap_outputs(results, folder, model_label):
    gap_df = observed_gap_by_decile(results, model_label)
    if gap_df is None:
        return None, None, None

    gap_df.to_csv(folder / 'uplift_observed_gap_by_decile.csv', index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    x_values = np.arange(len(gap_df))
    gaps = gap_df['observed_control_minus_treated_gap'].to_numpy(dtype=float)
    ci_lower = gap_df['observed_gap_ci_lower_95'].to_numpy(dtype=float)
    ci_upper = gap_df['observed_gap_ci_upper_95'].to_numpy(dtype=float)
    yerr = np.vstack([gaps - ci_lower, ci_upper - gaps])
    ax.bar(x_values, gaps, color='#4c78a8')
    ax.errorbar(x_values, gaps, yerr=yerr, fmt='none', ecolor='#222222', capsize=4, linewidth=1)
    ax.axhline(0, color='gray', linewidth=1)
    ax.set_xticks(x_values)
    ax.set_xticklabels(gap_df['uplift_decile'].astype(str))
    ax.set_title(f'{model_label}: Observed Control-Treated ED Gap by Uplift Decile')
    ax.set_xlabel('Uplift Decile: 1 = Highest Predicted Benefit')
    ax.set_ylabel('Observed Control ED Rate - Treated ED Rate')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_observed_gap_by_decile.png', dpi=150)
    plt.close(fig)

    curve_df = uplift_curve_by_decile(results, model_label)
    curve_summary = uplift_curve_summary(curve_df, model_label)
    if curve_df is not None:
        curve_df.to_csv(folder / 'uplift_curve_by_decile.csv', index=False)

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(
            curve_df['population_fraction_targeted'],
            curve_df['avg_predicted_benefit'],
            marker='o',
            label='Average predicted benefit',
        )
        ax.plot(
            curve_df['population_fraction_targeted'],
            curve_df['observed_control_minus_treated_gap'],
            marker='o',
            label='Observed control-treated gap',
        )
        ax.fill_between(
            curve_df['population_fraction_targeted'].to_numpy(dtype=float),
            curve_df['observed_gap_ci_lower_95'].to_numpy(dtype=float),
            curve_df['observed_gap_ci_upper_95'].to_numpy(dtype=float),
            color='#f58518',
            alpha=0.18,
            label='Observed gap 95% CI',
        )
        ax.axhline(0, color='gray', linewidth=1)
        ax.set_title(f'{model_label}: Cumulative Uplift Curve by Targeted Fraction')
        ax.set_xlabel('Cumulative Fraction Targeted')
        ax.set_ylabel('ED Rate Difference / Predicted Benefit')
        ax.legend()
        fig.tight_layout()
        fig.savefig(folder / 'dashboard_uplift_curve_by_decile.png', dpi=150)
        plt.close(fig)

    if curve_summary is not None:
        curve_summary.to_csv(folder / 'uplift_curve_summary.csv', index=False)

    print(f'{model_label} observed control-treated gap by uplift decile:')
    display(gap_df)
    print()
    print(f'{model_label} cumulative uplift curve by decile:')
    display(curve_df)
    print()
    if curve_summary is not None:
        print(f'{model_label} uplift curve summary:')
        display(curve_summary)
        print()
    return gap_df, curve_df, curve_summary


observed_gap_xgboost, uplift_curve_xgboost, uplift_curve_summary_xgboost = save_observed_gap_outputs(
    results_test_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None:
    observed_gap_glmnet, uplift_curve_glmnet, uplift_curve_summary_glmnet = save_observed_gap_outputs(
        results_test_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    observed_gap_glmnet = None
    uplift_curve_glmnet = None
    uplift_curve_summary_glmnet = None
    print('Skipped GLMNET observed gap outputs because GLMNET scoring was not available.')





# Apply the same observed within-decile treatment-control comparison to the
# XGBoost X-Learner and the overlap-weighted XGBoost T-Learner.
def save_observed_gap_decile_only(results, folder, model_label):
    gap_df = observed_gap_by_decile(results, model_label)
    if gap_df is None:
        return None

    gap_df.to_csv(folder / 'uplift_observed_gap_by_decile.csv', index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    x_values = np.arange(len(gap_df))
    gaps = gap_df['observed_control_minus_treated_gap'].to_numpy(dtype=float)
    ci_lower = gap_df['observed_gap_ci_lower_95'].to_numpy(dtype=float)
    ci_upper = gap_df['observed_gap_ci_upper_95'].to_numpy(dtype=float)
    yerr = np.vstack([gaps - ci_lower, ci_upper - gaps])
    bar_colors = np.where(gaps < 0, '#c44e52', '#4c78a8')

    ax.bar(x_values, gaps, color=bar_colors)
    ax.errorbar(
        x_values,
        gaps,
        yerr=yerr,
        fmt='none',
        ecolor='#222222',
        capsize=4,
        linewidth=1,
    )
    ax.axhline(0, color='gray', linewidth=1)
    ax.set_xticks(x_values)
    ax.set_xticklabels(gap_df['uplift_decile'].astype(str))
    ax.set_title(f'{model_label}: Observed Control-Treated ED Gap by Uplift Decile')
    ax.set_xlabel('Uplift Decile: 1 = Highest Predicted Benefit')
    ax.set_ylabel('Observed Control ED Rate - Treated ED Rate')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_observed_gap_by_decile.png', dpi=150)
    plt.close(fig)

    print(f'{model_label} observed control-treated ED gap by uplift decile:')
    display(gap_df)
    print()
    return gap_df


observed_gap_xlearner_xgboost = save_observed_gap_decile_only(
    results_test_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost X-Learner',
)
observed_gap_overlap_weighted_xgboost = save_observed_gap_decile_only(
    results_test_overlap_weighted_xgboost,
    overlap_weighted_xgboost_output_folder,
    'Overlap-Weighted XGBoost T-Learner',
)

observed_gap_comparison_columns = [
    'model',
    'uplift_decile',
    'n',
    'treated_n',
    'control_n',
    'treated_events',
    'control_events',
    'treated_observed_ed_rate',
    'control_observed_ed_rate',
    'observed_control_minus_treated_gap',
    'observed_gap_ci_lower_95',
    'observed_gap_ci_upper_95',
]
observed_gap_xgboost_model_comparison = pd.concat(
    [
        observed_gap_xgboost[observed_gap_comparison_columns],
        observed_gap_xlearner_xgboost[observed_gap_comparison_columns],
        observed_gap_overlap_weighted_xgboost[observed_gap_comparison_columns],
    ],
    ignore_index=True,
)
observed_gap_xgboost_model_comparison.to_csv(
    output_folder / 'xgboost_observed_gap_by_decile_comparison.csv',
    index=False,
)

print('Observed treatment-control ED difference by decile across XGBoost models:')
display(observed_gap_xgboost_model_comparison)
print(
    'Interpretation: positive values favor treatment; negative values indicate higher '
    'observed ED rates among treated members within that model-defined decile. '
    'Intervals crossing zero are inconclusive.'
)
print(
    'These are unadjusted observed associations within predicted deciles, not randomized '
    'causal effects. The confidence intervals reflect outcome sampling uncertainty only.'
)


---
## 17. VARIABLE IMPORTANCE
---


In [ ]:
importance_treated_xgboost = xgb_importance_frame(model_treated)
importance_control_xgboost = xgb_importance_frame(model_control)

# Backward-compatible aliases for the primary XGBoost importance tables.
importance_treated = importance_treated_xgboost
importance_control = importance_control_xgboost

print('Top variables in XGBoost treated model:')
display(importance_treated_xgboost.head(20))
print()

print('Top variables in XGBoost control model:')
display(importance_control_xgboost.head(20))
print()

if enet_treated is not None and enet_control is not None:
    glmnet_importance_treated = glmnet_contribution_importance_frame(enet_treated, x_test, 'Treated Model')
    glmnet_importance_control = glmnet_contribution_importance_frame(enet_control, x_test, 'Control Model')

    print('Top variables in GLMNET treated model:')
    display(glmnet_importance_treated.head(20))
    print()

    print('Top variables in GLMNET control model:')
    display(glmnet_importance_control.head(20))
    print()
else:
    glmnet_importance_treated = None
    glmnet_importance_control = None


---
## 18. SCORE FULL FILES
---


In [ ]:
full_x_df = model_df[feature_cols].copy()
full_prepared_df = prepare_predictor_frame(full_x_df)
full_matrix = design_matrix_from_prepared(full_prepared_df, combined_matrix.columns)

full_pred_treated_xgboost = model_treated.predict(make_dmatrix(full_matrix))
full_pred_control_xgboost = model_control.predict(make_dmatrix(full_matrix))
scored_full_xgboost = build_uplift_results(model_df, full_pred_treated_xgboost, full_pred_control_xgboost)

# Backward-compatible alias for the primary XGBoost scored file.
scored_full = scored_full_xgboost

if enet_treated is not None and enet_control is not None:
    full_pred_treated_glmnet = enet_treated['best_model'].predict_proba(full_matrix)[:, 1]
    full_pred_control_glmnet = enet_control['best_model'].predict_proba(full_matrix)[:, 1]
    scored_full_glmnet = build_uplift_results(model_df, full_pred_treated_glmnet, full_pred_control_glmnet)
else:
    full_pred_treated_glmnet = None
    full_pred_control_glmnet = None
    scored_full_glmnet = None


---
## 19. WRITE OUTPUTS
---


In [ ]:
scored_full_xgboost.to_csv(xgboost_output_path, index=False)
decile_summary_xgboost.to_csv(xgboost_summary_path, index=False)

print('XGBoost scored full file written to:', xgboost_output_path, '\n')
print('XGBoost decile summary written to:', xgboost_summary_path, '\n')

if scored_full_glmnet is not None and decile_summary_glmnet is not None:
    scored_full_glmnet.to_csv(glmnet_output_path, index=False)
    decile_summary_glmnet.to_csv(glmnet_summary_path, index=False)

    print('GLMNET scored full file written to:', glmnet_output_path, '\n')
    print('GLMNET decile summary written to:', glmnet_summary_path, '\n')
else:
    print('Skipped GLMNET core CSV outputs because GLMNET scoring was not available.\n')


---
## 20. INTERPRETATION
---


In [ ]:
print('INTERPRETATION:')
print('- pred_ed_if_treated = predicted probability of ED within 90d if treated')
print('- pred_ed_if_control = predicted probability of ED within 90d if not treated')
print('- benefit_score = pred_ed_if_control - pred_ed_if_treated')
print('- Higher benefit_score means treatment is predicted to reduce ED risk more')
print('- Uplift decile 1 = highest predicted treatment benefit')


---
## DASHBOARD VIEWS
---


In [ ]:
dashboard_folder = xgboost_output_folder


def save_bar_chart(df, x_col, y_col, title, x_label, y_label, path, width=8, height=5, y_max=None):
    fig, ax = plt.subplots(figsize=(width, height))
    ax.bar(df[x_col].astype(str), df[y_col])
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    if y_max is not None:
        ax.set_ylim(0, y_max)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def save_decile_dashboard_charts(decile_df, folder, model_label, benefit_y_max=None):
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'avg_benefit_score',
        f'{model_label}: Average Predicted Intervention Benefit by Uplift Decile',
        'Uplift Decile: 1 = Highest Predicted Benefit',
        'Average Benefit Score',
        folder / 'dashboard_avg_benefit_by_decile.png',
        y_max=benefit_y_max,
    )
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'observed_ed_rate',
        f'{model_label}: Observed 90-Day ED Rate by Uplift Decile',
        'Uplift Decile',
        'Observed ED Rate',
        folder / 'dashboard_observed_ed_rate_by_decile.png',
    )
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'treated_pct',
        f'{model_label}: Current Treatment Penetration by Uplift Decile',
        'Uplift Decile',
        'Percent Treated',
        folder / 'dashboard_treated_pct_by_decile.png',
    )

    decile_long = decile_df.melt(
        id_vars='uplift_decile',
        value_vars=['avg_pred_ed_if_treated', 'avg_pred_ed_if_control'],
        var_name='scenario',
        value_name='predicted_ed_rate',
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    scenarios = list(decile_long['scenario'].unique())
    x_values = np.arange(len(decile_df))
    bar_width = 0.35
    for offset, scenario in enumerate(scenarios):
        values = decile_long[decile_long['scenario'] == scenario]['predicted_ed_rate'].to_numpy()
        ax.bar(x_values + (offset - 0.5) * bar_width, values, width=bar_width, label=scenario)
    ax.set_xticks(x_values)
    ax.set_xticklabels(decile_df['uplift_decile'].astype(str))
    ax.set_title(f'{model_label}: Predicted ED Risk Treated vs Control by Decile')
    ax.set_xlabel('Uplift Decile')
    ax.set_ylabel('Predicted ED Rate')
    ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_predicted_treated_vs_control.png', dpi=150)
    plt.close(fig)


save_decile_dashboard_charts(decile_summary_xgboost, xgboost_output_folder, 'XGBoost')
print('XGBoost dashboard charts saved to:', xgboost_output_folder)

if decile_summary_glmnet is not None:
    save_decile_dashboard_charts(decile_summary_glmnet, glmnet_output_folder, 'GLMNET')
    print('GLMNET dashboard charts saved to:', glmnet_output_folder)
else:
    print('Skipped GLMNET dashboard charts because GLMNET scoring was not available.')


---
## ROI PER DECILE
---


In [ ]:
cost_per_ed_visit = 1200
cost_per_intervention = 250


def build_roi_summary(decile_df):
    roi_df = decile_df.copy()
    roi_df['expected_ed_rate_reduction'] = roi_df['avg_benefit_score']
    roi_df['expected_ed_visits_avoided'] = roi_df['n'] * roi_df['expected_ed_rate_reduction']
    roi_df['gross_savings'] = roi_df['expected_ed_visits_avoided'] * cost_per_ed_visit
    roi_df['intervention_cost'] = roi_df['n'] * cost_per_intervention
    roi_df['net_savings'] = roi_df['gross_savings'] - roi_df['intervention_cost']
    roi_df['roi'] = roi_df['net_savings'] / roi_df['intervention_cost']
    return roi_df


def save_roi_outputs(decile_df, folder, model_label):
    roi_df = build_roi_summary(decile_df)
    print(f'{model_label} ROI summary:')
    display(roi_df)
    roi_df.to_csv(folder / 'uplift_roi_by_decile.csv', index=False)
    save_bar_chart(
        roi_df,
        'uplift_decile',
        'net_savings',
        f'{model_label}: Estimated Net Savings by Uplift Decile',
        'Uplift Decile',
        'Estimated Net Savings',
        folder / 'dashboard_roi_net_savings_by_decile.png',
    )
    print(f'{model_label} ROI summary saved to:', folder)
    print()
    return roi_df


roi_summary_xgboost = save_roi_outputs(decile_summary_xgboost, xgboost_output_folder, 'XGBoost')

# Backward-compatible alias for the primary XGBoost ROI summary.
roi_summary = roi_summary_xgboost

if decile_summary_glmnet is not None:
    roi_summary_glmnet = save_roi_outputs(decile_summary_glmnet, glmnet_output_folder, 'GLMNET')
else:
    roi_summary_glmnet = None
    print('Skipped GLMNET ROI outputs because GLMNET scoring was not available.')


---
## WRITE-UP TOP BENEFIT DECILE SUMMARY
---


In [ ]:
def top_benefit_decile_summary(results, roi_df, model_label):
    if results is None or roi_df is None:
        return None

    top_decile = results[results['uplift_decile'] == 1].copy()
    if top_decile.empty:
        return None

    treated = top_decile[top_decile['intervention_flag'] == 1]
    control = top_decile[top_decile['intervention_flag'] == 0]
    treated_rate = treated['outcome_ed_90d'].mean() if not treated.empty else np.nan
    control_rate = control['outcome_ed_90d'].mean() if not control.empty else np.nan
    top_roi = roi_df.loc[roi_df['uplift_decile'] == 1].iloc[0]

    return pd.DataFrame(
        [
            {
                'model': model_label,
                'top_decile_n': len(top_decile),
                'top_decile_treated_n': len(treated),
                'top_decile_control_n': len(control),
                'top_decile_avg_predicted_benefit': top_decile['benefit_score'].mean(),
                'top_decile_observed_ed_rate': top_decile['outcome_ed_90d'].mean(),
                'top_decile_treated_observed_ed_rate': treated_rate,
                'top_decile_control_observed_ed_rate': control_rate,
                'top_decile_observed_control_minus_treated_gap': control_rate - treated_rate,
                'top_decile_treated_pct': top_decile['intervention_flag'].mean(),
                'top_decile_estimated_ed_visits_avoided': top_roi['expected_ed_visits_avoided'],
                'top_decile_gross_savings': top_roi['gross_savings'],
                'top_decile_intervention_cost': top_roi['intervention_cost'],
                'top_decile_net_savings': top_roi['net_savings'],
                'top_decile_roi': top_roi['roi'],
            }
        ]
    )


def save_top_decile_summary(results, roi_df, folder, model_label):
    summary = top_benefit_decile_summary(results, roi_df, model_label)
    if summary is None:
        return None
    summary.to_csv(folder / 'top_benefit_decile_summary.csv', index=False)
    print(f'{model_label} top benefit decile summary:')
    display(summary)
    print()
    return summary


top_decile_summary_xgboost = save_top_decile_summary(
    results_test_xgboost,
    roi_summary_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None and roi_summary_glmnet is not None:
    top_decile_summary_glmnet = save_top_decile_summary(
        results_test_glmnet,
        roi_summary_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    top_decile_summary_glmnet = None
    print('Skipped GLMNET top benefit decile summary because GLMNET scoring or ROI was not available.')


---
## RISK MODEL DRIVER OUTPUTS
---


In [ ]:
def save_model_driver_outputs(
    treated_importance,
    control_importance,
    folder,
    model_label,
    value_col,
    x_label,
    csv_name='shap_importance_treated_control_models.csv',
):
    combined = pd.concat([treated_importance, control_importance], ignore_index=True)
    combined.to_csv(folder / csv_name, index=False)

    for driver_df, treatment_label, filename in [
        (treated_importance, 'Treated Model', 'dashboard_shap_treated_model.png'),
        (control_importance, 'Control Model', 'dashboard_shap_control_model.png'),
    ]:
        top = driver_df.nlargest(20, value_col).sort_values(value_col)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(top['feature'], top[value_col])
        ax.set_title(f'{model_label}: Top Risk Drivers - {treatment_label}')
        ax.set_xlabel(x_label)
        ax.set_ylabel('Feature')
        fig.tight_layout()
        fig.savefig(folder / filename, dpi=150)
        plt.close(fig)

    print(f'{model_label} risk-driver outputs saved to:', folder)
    return combined


shap_treated_importance = shap_importance_frame(model_treated, x_test, 'Treated Model')
shap_control_importance = shap_importance_frame(model_control, x_test, 'Control Model')
shap_importance_combined = save_model_driver_outputs(
    shap_treated_importance,
    shap_control_importance,
    xgboost_output_folder,
    'XGBoost',
    'mean_abs_shap',
    'Mean Absolute SHAP Contribution',
)

if enet_treated is not None and enet_control is not None:
    # GLMNET does not use XGBoost SHAP contributions here, so save an analogous
    # standardized logit-contribution driver artifact in the GLMNET folder.
    glmnet_driver_treated = glmnet_contribution_importance_frame(enet_treated, x_test, 'Treated Model')
    glmnet_driver_control = glmnet_contribution_importance_frame(enet_control, x_test, 'Control Model')
    glmnet_driver_combined = save_model_driver_outputs(
        glmnet_driver_treated,
        glmnet_driver_control,
        glmnet_output_folder,
        'GLMNET',
        'mean_abs_model_contribution',
        'Mean Absolute Standardized Logit Contribution',
    )
else:
    glmnet_driver_treated = None
    glmnet_driver_control = None
    glmnet_driver_combined = None
    print('Skipped GLMNET model-driver outputs because GLMNET scoring was not available.')

print('\nFiles currently in XGBoost output folder:')
print('\n'.join(str(path) for path in sorted(xgboost_output_folder.iterdir())))

print('\nFiles currently in GLMNET output folder:')
print('\n'.join(str(path) for path in sorted(glmnet_output_folder.iterdir())))


---
## BENEFIT SCORE DRIVER IMPORTANCE
---


In [ ]:
def save_benefit_driver_outputs(importance_df, folder, model_label, value_col, x_label):
    importance_df.to_csv(
        folder / 'shap_importance_benefit_score.csv',
        index=False,
    )

    top = importance_df.head(20).sort_values(value_col)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top[value_col])
    ax.set_title(f'{model_label}: Top Drivers of Predicted Treatment Benefit')
    ax.set_xlabel(x_label)
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_shap_benefit_score.png', dpi=150)
    plt.close(fig)

    print(f'{model_label} benefit-driver outputs saved to:', folder)
    print()


def xgboost_benefit_shap_importance_frame(model_treated, model_control, x_matrix):
    benefit_dtest = make_dmatrix(x_matrix)

    control_contribs = model_control.predict(benefit_dtest, pred_contribs=True)
    treated_contribs = model_treated.predict(benefit_dtest, pred_contribs=True)
    shap_columns = [*x_matrix.columns, 'BIAS']

    control_shap = pd.DataFrame(control_contribs, columns=shap_columns, index=x_matrix.index)
    treated_shap = pd.DataFrame(treated_contribs, columns=shap_columns, index=x_matrix.index)

    # Signed contribution to model-estimated benefit on the XGBoost raw-margin scale.
    benefit_shap = control_shap - treated_shap
    benefit_shap_no_bias = benefit_shap.drop(columns=['BIAS'], errors='ignore')

    return (
        pd.DataFrame(
            {
                'feature': benefit_shap_no_bias.columns,
                'mean_abs_benefit_shap': benefit_shap_no_bias.abs().mean(axis=0).to_numpy(),
                'mean_signed_benefit_shap': benefit_shap_no_bias.mean(axis=0).to_numpy(),
                'pct_positive_benefit_shap': benefit_shap_no_bias.gt(0).mean(axis=0).to_numpy(),
                'importance_type': 'xgboost_raw_margin_shap_difference',
            }
        )
        .sort_values('mean_abs_benefit_shap', ascending=False)
        .reset_index(drop=True)
    )


def glmnet_benefit_contribution_importance_frame(treated_model_info, control_model_info, x_matrix):
    treated_pipeline = treated_model_info['best_model']
    control_pipeline = control_model_info['best_model']

    treated_scaler = treated_pipeline.named_steps['standardscaler']
    control_scaler = control_pipeline.named_steps['standardscaler']
    treated_model = treated_pipeline.named_steps['logisticregressioncv']
    control_model = control_pipeline.named_steps['logisticregressioncv']

    treated_contrib = pd.DataFrame(
        treated_scaler.transform(x_matrix) * treated_model.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )
    control_contrib = pd.DataFrame(
        control_scaler.transform(x_matrix) * control_model.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )

    # Signed contribution to model-estimated benefit on the GLMNET logit scale.
    benefit_contrib = control_contrib - treated_contrib

    return (
        pd.DataFrame(
            {
                'feature': benefit_contrib.columns,
                'mean_abs_benefit_contribution': benefit_contrib.abs().mean(axis=0).to_numpy(),
                'mean_signed_benefit_contribution': benefit_contrib.mean(axis=0).to_numpy(),
                'pct_positive_benefit_contribution': benefit_contrib.gt(0).mean(axis=0).to_numpy(),
                'control_coefficient': control_model.coef_.ravel(),
                'treated_coefficient': treated_model.coef_.ravel(),
                'coefficient_difference': control_model.coef_.ravel() - treated_model.coef_.ravel(),
                'importance_type': 'glmnet_standardized_logit_contribution_difference',
            }
        )
        .sort_values('mean_abs_benefit_contribution', ascending=False)
        .reset_index(drop=True)
    )


xgboost_benefit_shap_importance = xgboost_benefit_shap_importance_frame(
    model_treated,
    model_control,
    x_test,
)

print('Top XGBoost SHAP features driving predicted treatment benefit:')
display(xgboost_benefit_shap_importance.head(20))

save_benefit_driver_outputs(
    xgboost_benefit_shap_importance,
    xgboost_output_folder,
    'XGBoost',
    'mean_abs_benefit_shap',
    'Mean Absolute SHAP Contribution to Benefit',
)

if enet_treated is not None and enet_control is not None:
    glmnet_benefit_importance = glmnet_benefit_contribution_importance_frame(
        enet_treated,
        enet_control,
        x_test,
    )

    print('Top GLMNET features driving predicted treatment benefit:')
    display(glmnet_benefit_importance.head(20))

    save_benefit_driver_outputs(
        glmnet_benefit_importance,
        glmnet_output_folder,
        'GLMNET',
        'mean_abs_benefit_contribution',
        'Mean Absolute Standardized Logit Contribution to Benefit',
    )
else:
    glmnet_benefit_importance = None
    print('Skipped GLMNET benefit-driver outputs because GLMNET scoring was not available.')


---
## X-LEARNER BENEFIT SCORE DRIVER IMPORTANCE
---

This section creates GLMNet X-learner benefit-driver outputs so Task 6 can compare the T-learner and X-learner benefit-driver patterns side by side.


In [ ]:

def glmnet_xlearner_benefit_contribution_importance_frame(tau_treated_model, tau_control_model, propensity_model, x_matrix):
    treated_scaler = tau_treated_model.named_steps['standardscaler']
    control_scaler = tau_control_model.named_steps['standardscaler']
    if treated_scaler is not control_scaler:
        raise ValueError('GLMNET X-learner treated/control effect models must share the same scaler for contribution attribution.')
    shared_x = treated_scaler.transform(x_matrix)
    treated_reg = tau_treated_model.named_steps['elasticnetcv']
    control_reg = tau_control_model.named_steps['elasticnetcv']

    propensity = clipped_propensity(propensity_model, x_matrix)
    treated_weight = 1 - propensity
    control_weight = propensity

    treated_effect_contrib = pd.DataFrame(
        shared_x * treated_reg.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )
    control_effect_contrib = pd.DataFrame(
        shared_x * control_reg.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )

    weighted_benefit_contrib = treated_effect_contrib.mul(treated_weight, axis=0) + control_effect_contrib.mul(control_weight, axis=0)

    return (
        pd.DataFrame(
            {
                'feature': weighted_benefit_contrib.columns,
                'mean_abs_xlearner_benefit_contribution': weighted_benefit_contrib.abs().mean(axis=0).to_numpy(),
                'mean_signed_xlearner_benefit_contribution': weighted_benefit_contrib.mean(axis=0).to_numpy(),
                'pct_positive_xlearner_benefit_contribution': weighted_benefit_contrib.gt(0).mean(axis=0).to_numpy(),
                'treated_effect_coefficient': treated_reg.coef_.ravel(),
                'control_effect_coefficient': control_reg.coef_.ravel(),
                'avg_treated_effect_weight': treated_weight.mean(),
                'avg_control_effect_weight': control_weight.mean(),
                'importance_type': 'glmnet_xlearner_weighted_shared_standardized_effect_contribution',
            }
        )
        .sort_values('mean_abs_xlearner_benefit_contribution', ascending=False)
        .reset_index(drop=True)
    )


def save_xlearner_glmnet_benefit_driver_outputs(importance_df, folder):
    importance_path = folder / 'xlearner_benefit_driver_importance.csv'
    importance_df.to_csv(importance_path, index=False)

    # Also save to the common benefit-driver filename so README generation can use parallel paths.
    importance_df.to_csv(folder / 'shap_importance_benefit_score.csv', index=False)

    top = importance_df.head(20).sort_values('mean_abs_xlearner_benefit_contribution')
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top['mean_abs_xlearner_benefit_contribution'])
    ax.set_title('GLMNET X-Learner: Top Drivers of Predicted Treatment Benefit')
    ax.set_xlabel('Mean Absolute Weighted Standardized Effect Contribution')
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_xlearner_benefit_drivers.png', dpi=150)
    fig.savefig(folder / 'dashboard_shap_benefit_score.png', dpi=150)
    plt.close(fig)

    print('GLMNET X-learner benefit-driver outputs saved to:', folder)
    print('X-learner benefit-driver CSV:', importance_path)


def save_t_vs_x_benefit_driver_comparison_chart(t_importance, x_importance, folder):
    if t_importance is None or x_importance is None:
        print('Skipped T-learner versus X-learner benefit-driver comparison chart because one input is missing.')
        return None

    t_top = t_importance.head(15).sort_values('mean_abs_benefit_contribution')
    x_top = x_importance.head(15).sort_values('mean_abs_xlearner_benefit_contribution')

    fig, axes = plt.subplots(ncols=2, figsize=(14, 6))
    axes[0].barh(t_top['feature'], t_top['mean_abs_benefit_contribution'])
    axes[0].set_title('GLMNET T-Learner')
    axes[0].set_xlabel('Mean Absolute Benefit Contribution')
    axes[0].set_ylabel('Feature')

    axes[1].barh(x_top['feature'], x_top['mean_abs_xlearner_benefit_contribution'])
    axes[1].set_title('GLMNET X-Learner')
    axes[1].set_xlabel('Mean Absolute Benefit Contribution')
    axes[1].set_ylabel('Feature')

    fig.suptitle('Top Drivers of Predicted Treatment Benefit')
    fig.tight_layout()
    chart_path = folder / 'dashboard_t_vs_x_benefit_driver_comparison.png'
    fig.savefig(chart_path, dpi=150)
    plt.close(fig)
    print('T-learner versus X-learner benefit-driver comparison chart saved to:', chart_path)
    return chart_path


if (
    'glmnet_tau_treated_model' in globals()
    and 'glmnet_tau_control_model' in globals()
    and glmnet_tau_treated_model is not None
    and glmnet_tau_control_model is not None
):
    glmnet_xlearner_benefit_importance = glmnet_xlearner_benefit_contribution_importance_frame(
        glmnet_tau_treated_model,
        glmnet_tau_control_model,
        propensity_model,
        x_test,
    )
    print('Top GLMNET X-learner features driving predicted treatment benefit:')
    display(glmnet_xlearner_benefit_importance.head(20))

    save_xlearner_glmnet_benefit_driver_outputs(
        glmnet_xlearner_benefit_importance,
        xlearner_glmnet_output_folder,
    )
    save_t_vs_x_benefit_driver_comparison_chart(
        glmnet_benefit_importance,
        glmnet_xlearner_benefit_importance,
        xlearner_glmnet_output_folder,
    )
else:
    glmnet_xlearner_benefit_importance = None
    print('Skipped GLMNET X-learner benefit-driver outputs because GLMNET X-learner models were not available.')


---
## GLMNET BENEFIT SCORE SHAP CONTRIBUTIONS

This section adds SHAP-based global explanations for the final GLMNet T-learner and X-learner benefit-score functions. Mean absolute SHAP ranks global importance, while signed SHAP values show average positive and negative contribution direction.


In [ ]:
def _as_feature_frame(values, feature_columns):
    if isinstance(values, pd.DataFrame):
        return values.loc[:, feature_columns]
    return pd.DataFrame(values, columns=feature_columns)


def _shap_expected_value(explanation):
    base_values = np.asarray(explanation.base_values, dtype=float)
    if base_values.ndim == 0:
        return float(base_values)
    return float(np.nanmean(base_values))


def _shap_values_frame(explanation, index, feature_columns):
    shap_array = np.asarray(explanation.values, dtype=float)
    if shap_array.ndim == 3:
        shap_array = shap_array[:, :, 0]
    return pd.DataFrame(shap_array, columns=feature_columns, index=index)


def _summarize_benefit_shap(shap_df, feature_columns, importance_type):
    positive_only = shap_df.where(shap_df > 0, 0.0)
    negative_only = shap_df.where(shap_df < 0, 0.0)

    return (
        pd.DataFrame(
            {
                "feature": feature_columns,
                "mean_abs_benefit_shap": shap_df.abs().mean(axis=0).to_numpy(),
                "mean_signed_benefit_shap": shap_df.mean(axis=0).to_numpy(),
                "mean_positive_benefit_shap": positive_only.mean(axis=0).to_numpy(),
                "mean_negative_benefit_shap": negative_only.mean(axis=0).to_numpy(),
                "pct_positive_benefit_shap": shap_df.gt(0).mean(axis=0).to_numpy(),
                "pct_negative_benefit_shap": shap_df.lt(0).mean(axis=0).to_numpy(),
                "importance_type": importance_type,
            }
        )
        .sort_values("mean_abs_benefit_shap", ascending=False)
        .reset_index(drop=True)
    )


def _save_global_benefit_shap_outputs(
    importance_df,
    shap_df,
    reconciliation_df,
    folder,
    prefix,
    model_label,
):
    importance_path = folder / f"{prefix}_global_benefit_shap_importance.csv"
    member_path = folder / f"{prefix}_member_benefit_shap_values.csv"
    reconciliation_path = folder / f"{prefix}_global_benefit_shap_reconciliation.csv"
    chart_path = folder / f"dashboard_{prefix}_global_benefit_shap.png"

    importance_df.to_csv(importance_path, index=False)
    shap_df.to_csv(member_path, index=True, index_label="row_index")
    reconciliation_df.to_csv(reconciliation_path, index=False)

    top = importance_df.head(20).sort_values("mean_abs_benefit_shap")

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top["feature"], top["mean_abs_benefit_shap"])
    ax.set_title(f"{model_label}: Global SHAP Drivers of Benefit Score")
    ax.set_xlabel("Mean absolute SHAP contribution to benefit score")
    ax.set_ylabel("Feature")
    fig.tight_layout()
    fig.savefig(chart_path, dpi=150)
    plt.close(fig)

    print(f"{model_label} global benefit SHAP outputs saved to:", folder)
    print("Importance:", importance_path)
    print("Member SHAP values:", member_path)
    print("Reconciliation:", reconciliation_path)
    print("Chart:", chart_path)


def _run_global_benefit_shap(
    benefit_predict_fn,
    background_x,
    explain_x,
    expected_benefit,
    folder,
    prefix,
    model_label,
    seed=123,
):
    if shap is None:
        print(
            f"Skipped {model_label} global benefit SHAP because the shap package is not installed. "
            "Install with: pip install shap"
        )
        return None, None, None

    feature_columns = list(explain_x.columns)
    background_sample = shap.sample(
        background_x,
        min(100, len(background_x)),
        random_state=seed,
    )

    def predict_for_shap(values):
        values_df = _as_feature_frame(values, feature_columns)
        return np.asarray(benefit_predict_fn(values_df), dtype=float)

    masker = shap.maskers.Independent(background_sample)
    explainer = shap.Explainer(
        predict_for_shap,
        masker,
        algorithm="permutation",
    )

    explanation = explainer(
        explain_x,
        max_evals=2 * len(feature_columns) + 1,
    )

    shap_df = _shap_values_frame(explanation, explain_x.index, feature_columns)

    importance_df = _summarize_benefit_shap(
        shap_df,
        feature_columns,
        f"{prefix}_final_benefit_score_permutation_shap",
    )

    baseline_value = _shap_expected_value(explanation)
    signed_sum = float(importance_df["mean_signed_benefit_shap"].sum())
    reconstructed_average = baseline_value + signed_sum
    average_predicted_benefit = float(np.mean(expected_benefit))

    reconciliation_df = pd.DataFrame(
        [
            {
                "metric": "shap_baseline_benefit",
                "value": baseline_value,
            },
            {
                "metric": "sum_mean_signed_feature_shap",
                "value": signed_sum,
            },
            {
                "metric": "baseline_plus_signed_shap_sum",
                "value": reconstructed_average,
            },
            {
                "metric": "average_predicted_benefit_from_model",
                "value": average_predicted_benefit,
            },
            {
                "metric": "reconciliation_difference",
                "value": reconstructed_average - average_predicted_benefit,
            },
        ]
    )

    _save_global_benefit_shap_outputs(
        importance_df,
        shap_df,
        reconciliation_df,
        folder,
        prefix,
        model_label,
    )

    display(importance_df.head(20))
    display(reconciliation_df)

    return importance_df, shap_df, reconciliation_df


glmnet_tlearner_global_shap_importance = None
glmnet_xlearner_global_shap_importance = None

# Use the encoded training design matrix as the SHAP background/reference population.
# Your notebook creates x_treated, x_control, and x_test, not x_train.
glmnet_shap_background_x = pd.concat([x_treated, x_control], axis=0)


if enet_treated is not None and enet_control is not None:
    treated_glmnet_pipeline = enet_treated["best_model"]
    control_glmnet_pipeline = enet_control["best_model"]

    def glmnet_tlearner_benefit_predict(frame):
        pred_treated = treated_glmnet_pipeline.predict_proba(frame)[:, 1]
        pred_control = control_glmnet_pipeline.predict_proba(frame)[:, 1]
        return pred_control - pred_treated

    glmnet_tlearner_expected_benefit = glmnet_tlearner_benefit_predict(x_test)

    (
        glmnet_tlearner_global_shap_importance,
        glmnet_tlearner_member_shap_values,
        glmnet_tlearner_shap_reconciliation,
    ) = _run_global_benefit_shap(
        glmnet_tlearner_benefit_predict,
        glmnet_shap_background_x,
        x_test,
        glmnet_tlearner_expected_benefit,
        glmnet_output_folder,
        "glmnet_tlearner",
        "GLMNET T-Learner",
        seed=123,
    )
else:
    print("Skipped GLMNET T-learner global benefit SHAP because GLMNET models were not available.")


if (
    "glmnet_tau_treated_model" in globals()
    and "glmnet_tau_control_model" in globals()
    and glmnet_tau_treated_model is not None
    and glmnet_tau_control_model is not None
):
    def glmnet_xlearner_benefit_predict(frame):
        treated_effect = glmnet_tau_treated_model.predict(frame)
        control_effect = glmnet_tau_control_model.predict(frame)
        propensity = clipped_propensity(propensity_model, frame)
        return (1 - propensity) * treated_effect + propensity * control_effect

    glmnet_xlearner_expected_benefit = glmnet_xlearner_benefit_predict(x_test)

    (
        glmnet_xlearner_global_shap_importance,
        glmnet_xlearner_member_shap_values,
        glmnet_xlearner_shap_reconciliation,
    ) = _run_global_benefit_shap(
        glmnet_xlearner_benefit_predict,
        glmnet_shap_background_x,
        x_test,
        glmnet_xlearner_expected_benefit,
        xlearner_glmnet_output_folder,
        "glmnet_xlearner",
        "GLMNET X-Learner",
        seed=123,
    )
else:
    print("Skipped GLMNET X-learner global benefit SHAP because GLMNET X-learner models were not available.")


---
## WRITE-UP MODEL EVALUATION SUMMARY
---


In [ ]:
def get_group_metric(df, group, metric_col):
    if df is None:
        return np.nan
    match = df[df['group'] == group]
    if match.empty:
        return np.nan
    return float(match[metric_col].iloc[0])


def get_top_gap_metric(gap_df, metric_col):
    if gap_df is None or gap_df.empty:
        return np.nan
    match = gap_df[gap_df['uplift_decile'] == 1]
    if match.empty:
        return np.nan
    return match[metric_col].iloc[0]


def get_curve_metric(curve_summary_df, metric_col):
    if curve_summary_df is None or curve_summary_df.empty:
        return np.nan
    return float(curve_summary_df[metric_col].iloc[0])


def decile_gap_rank_correlation(gap_df):
    if gap_df is None or gap_df.empty:
        return np.nan
    valid = gap_df[['avg_predicted_benefit', 'observed_control_minus_treated_gap']].dropna()
    if len(valid) < 2:
        return np.nan
    return valid['avg_predicted_benefit'].corr(valid['observed_control_minus_treated_gap'], method='spearman')


def mean_abs_predicted_observed_gap_difference(gap_df):
    if gap_df is None or gap_df.empty:
        return np.nan
    return float(gap_df['abs_predicted_minus_observed_gap'].mean())


def build_model_evaluation_row(model_label, treated_cv_auc, control_cv_auc, treated_test_auc, control_test_auc, brier_df, calibration_summary, gap_df, curve_summary_df):
    return {
        'model': model_label,
        'treated_cv_auc': treated_cv_auc,
        'control_cv_auc': control_cv_auc,
        'treated_test_auc': treated_test_auc,
        'control_test_auc': control_test_auc,
        'treated_brier_score': get_group_metric(brier_df, 'Treated', 'brier_score'),
        'control_brier_score': get_group_metric(brier_df, 'Control', 'brier_score'),
        'treated_calibration_error': get_group_metric(calibration_summary, 'Treated', 'weighted_mean_abs_calibration_error'),
        'control_calibration_error': get_group_metric(calibration_summary, 'Control', 'weighted_mean_abs_calibration_error'),
        'top_decile_avg_predicted_benefit': get_top_gap_metric(gap_df, 'avg_predicted_benefit'),
        'top_decile_observed_control_minus_treated_gap': get_top_gap_metric(gap_df, 'observed_control_minus_treated_gap'),
        'top_decile_observed_gap_ci_lower_95': get_top_gap_metric(gap_df, 'observed_gap_ci_lower_95'),
        'top_decile_observed_gap_ci_upper_95': get_top_gap_metric(gap_df, 'observed_gap_ci_upper_95'),
        'top_decile_predicted_benefit_within_observed_gap_ci_95': get_top_gap_metric(gap_df, 'predicted_benefit_within_observed_gap_ci_95'),
        'benefit_gap_spearman_corr_by_decile': decile_gap_rank_correlation(gap_df),
        'mean_abs_predicted_minus_observed_gap_by_decile': mean_abs_predicted_observed_gap_difference(gap_df),
        'observed_gap_area_under_curve': get_curve_metric(curve_summary_df, 'observed_gap_area_under_curve'),
        'predicted_benefit_area_under_curve': get_curve_metric(curve_summary_df, 'predicted_benefit_area_under_curve'),
    }


model_evaluation_rows = [
    build_model_evaluation_row(
        'XGBoost',
        xgb_treated_cv['best_cv_auc'],
        xgb_control_cv['best_cv_auc'],
        auc_treated_cv_xgb,
        auc_control_cv_xgb,
        brier_xgboost,
        calibration_summary_xgboost,
        observed_gap_xgboost,
        uplift_curve_summary_xgboost,
    )
]

if enet_treated is not None and enet_control is not None:
    model_evaluation_rows.append(
        build_model_evaluation_row(
            'GLMNET',
            enet_treated['best_auc'],
            enet_control['best_auc'],
            auc_treated_cv_glmnet,
            auc_control_cv_glmnet,
            brier_glmnet,
            calibration_summary_glmnet,
            observed_gap_glmnet,
            uplift_curve_summary_glmnet,
        )
    )

model_evaluation_summary = pd.DataFrame(model_evaluation_rows)
model_evaluation_summary_path = output_folder / 'model_evaluation_summary.csv'
model_evaluation_summary.to_csv(model_evaluation_summary_path, index=False)

print('Model evaluation summary:')
display(model_evaluation_summary)
print('Model evaluation summary written to:', model_evaluation_summary_path)




---
## WRITE-UP MODEL RECOMMENDATION SUMMARY
---


In [ ]:
def auc_strength(value):
    if pd.isna(value):
        return 'Unavailable'
    if value >= 0.90:
        return 'Excellent'
    if value >= 0.80:
        return 'Strong'
    if value >= 0.70:
        return 'Acceptable'
    if value >= 0.60:
        return 'Weak'
    return 'Poor'


def calibration_label(row):
    avg_error = np.nanmean([row['treated_calibration_error'], row['control_calibration_error']])
    avg_brier = np.nanmean([row['treated_brier_score'], row['control_brier_score']])
    if pd.isna(avg_error):
        return 'Unavailable'
    if avg_error <= 0.02:
        return f'Well calibrated; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'
    if avg_error <= 0.05:
        return f'Moderately calibrated; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'
    return f'Calibration needs review; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'


def benefit_ranking_label(row):
    benefit = row['top_decile_avg_predicted_benefit']
    gap = row['top_decile_observed_control_minus_treated_gap']
    ci_lower = row['top_decile_observed_gap_ci_lower_95']
    ci_upper = row['top_decile_observed_gap_ci_upper_95']
    benefit_within_ci = row['top_decile_predicted_benefit_within_observed_gap_ci_95']
    rank_corr = row['benefit_gap_spearman_corr_by_decile']
    if pd.isna(benefit):
        return 'Unavailable'
    ci_text = f'95% CI {ci_lower:.3f} to {ci_upper:.3f}' if pd.notna(ci_lower) else '95% CI unavailable'
    if pd.isna(benefit_within_ci):
        within_text = 'predicted benefit CI check unavailable'
    elif bool(benefit_within_ci):
        within_text = 'predicted benefit is within observed-gap CI'
    else:
        within_text = 'predicted benefit is outside observed-gap CI'
    corr_text = f'; decile rank corr {rank_corr:.3f}' if pd.notna(rank_corr) else ''
    if benefit > 0 and gap > 0:
        return f'Top decile predicted benefit {benefit:.3f}; observed gap {gap:.3f} ({ci_text}); {within_text}{corr_text}'
    if benefit > 0:
        return f'Top decile predicted benefit {benefit:.3f}; observed gap not positive ({gap:.3f}, {ci_text}); {within_text}{corr_text}'
    return f'Top decile predicted benefit is not positive ({benefit:.3f}); observed gap {gap:.3f} ({ci_text}); {within_text}{corr_text}'


recommendation_rows = []
for _, row in model_evaluation_summary.iterrows():
    avg_auc = np.nanmean([row['treated_test_auc'], row['control_test_auc']])
    recommendation_rows.append(
        {
            'model': row['model'],
            'risk_prediction_strength': f'{auc_strength(avg_auc)} average test AUC ({avg_auc:.3f})',
            'probability_calibration': calibration_label(row),
            'benefit_ranking_quality': benefit_ranking_label(row),
            'explainability': 'Tree SHAP risk and benefit drivers' if row['model'] == 'XGBoost' else 'Standardized coefficient/logit contribution risk and benefit drivers',
            'model_evaluation_takeaway': 'Prioritization candidate if AUC, calibration, observed-gap direction, and uncertainty are acceptable; ROI is evaluated separately in the business value section.',
        }
    )

model_recommendation_summary = pd.DataFrame(recommendation_rows)
model_recommendation_summary_path = output_folder / 'model_recommendation_summary.csv'
model_recommendation_summary.to_csv(model_recommendation_summary_path, index=False)

print('Model recommendation summary:')
display(model_recommendation_summary)
print('Model recommendation summary written to:', model_recommendation_summary_path)





---
## RISK TIER VERSUS BENEFIT GROUP OUTPUTS
---


---
## SYNTHETIC TRUE-BENEFIT VALIDATION — NOT APPLICABLE

Oregon 2-Year does not provide known counterfactual treatment-effect ground truth.
Synthetic true-benefit calculations are intentionally not performed or fabricated.
This exception is recorded in `output_parity_manifest.csv`.


In [ ]:
synthetic_true_benefit_validation_status = pd.DataFrame([{
    'section': 'synthetic_true_benefit_validation',
    'applicability': 'not_applicable',
    'reason': 'Oregon 2-Year has no known counterfactual treatment-effect ground truth.',
}])
display(synthetic_true_benefit_validation_status)


In [ ]:
risk_tier_order = ['1', '2', '3', '4', '5']
risk_tier_display = {
    '1': 'Tier 1 (lowest risk)',
    '2': 'Tier 2',
    '3': 'Tier 3',
    '4': 'Tier 4',
    '5': 'Tier 5 (highest risk)',
}
benefit_group_order = ['High benefit', 'Medium benefit', 'Low benefit']

benefit_group_colors = {
    'High benefit': '#1f77b4',
    'Medium benefit': '#70ad47',
    'Low benefit': '#a6a6a6',
}

risk_tier_thresholds = pd.DataFrame(
    [
        {
            'risk_tier': tier,
            'risk_tier_label': risk_tier_display[tier],
            'current_risk_score_rule': 'Not derived from current_risk_score',
            'source_definition': 'Original Oregon 2-Year risk_tier assignment',
        }
        for tier in risk_tier_order
    ]
)

risk_tier_thresholds.to_csv(output_folder / 'risk_tier_thresholds.csv', index=False)

if scored_full_glmnet is None:
    raise ValueError(
        'GLMNet full scored output is unavailable. '
        'Run the GLMNet scoring cells before this risk-tier section.'
    )

if results_test_glmnet is None:
    raise ValueError(
        'GLMNet T-learner test output is unavailable. '
        'Run the GLMNet test scoring cells before this risk-tier section.'
    )

if results_test_xlearner_glmnet is None:
    raise ValueError(
        'GLMNet X-learner test output is unavailable. '
        'Run the X-learner cells before this risk-tier section.'
    )

valid_risk_tier_population = scored_full_glmnet.loc[
    scored_full_glmnet['risk_tier'].isin(risk_tier_order)
].copy()
unassigned_risk_tier_members = int(len(scored_full_glmnet) - len(valid_risk_tier_population))

risk_tier_population_summary = (
    valid_risk_tier_population.groupby('risk_tier', observed=False)
    .agg(
        members=('risk_tier', 'size'),
        pct_population=(
            'risk_tier',
            lambda s: len(s) / len(valid_risk_tier_population),
        ),
        min_current_risk_score=('current_risk_score', 'min'),
        max_current_risk_score=('current_risk_score', 'max'),
        avg_current_risk_score=('current_risk_score', 'mean'),
    )
    .reindex(risk_tier_order)
    .reset_index()
)
risk_tier_population_summary.insert(
    1,
    'risk_tier_label',
    risk_tier_population_summary['risk_tier'].map(risk_tier_display),
)

risk_tier_population_summary.to_csv(
    output_folder / 'risk_tier_population_summary.csv',
    index=False,
)


def assign_model_relative_benefit_groups(benefit_score):
    # Rank first so ties are resolved deterministically and qcut produces
    # three equal-frequency groups (sizes differ by at most one member).
    benefit_rank = benefit_score.rank(method='first', ascending=False)
    return pd.qcut(
        benefit_rank,
        q=3,
        labels=benefit_group_order,
    )


def build_risk_tier_benefit_group_outputs(
    scored_df,
    output_dir,
    file_prefix,
    chart_title,
):
    df = scored_df.copy()

    df['benefit_group'] = assign_model_relative_benefit_groups(
        df['benefit_score']
    )

    df = df.loc[df['risk_tier'].isin(risk_tier_order)].copy()

    df['risk_tier'] = pd.Categorical(
        df['risk_tier'],
        categories=risk_tier_order,
        ordered=True,
    )

    df['benefit_group'] = pd.Categorical(
        df['benefit_group'],
        categories=benefit_group_order,
        ordered=True,
    )

    summary = (
        df.groupby(['risk_tier', 'benefit_group'], observed=False)
        .size()
        .reset_index(name='members')
    )

    tier_totals = (
        df.groupby('risk_tier', observed=False)
        .size()
        .rename('risk_tier_members')
        .reset_index()
    )

    summary = summary.merge(tier_totals, on='risk_tier', how='left')

    summary['pct_within_risk_tier'] = np.where(
        summary['risk_tier_members'] > 0,
        summary['members'] / summary['risk_tier_members'],
        np.nan,
    )

    summary['benefit_group_definition'] = summary['benefit_group'].map(
        {
            'High benefit': 'predicted-benefit tercile (top third)',
            'Medium benefit': 'predicted-benefit tercile (middle third)',
            'Low benefit': 'predicted-benefit tercile (bottom third)',
        }
    )

    summary_path = output_dir / f'{file_prefix}_risk_tier_benefit_group_summary.csv'
    summary.to_csv(summary_path, index=False)

    pct = (
        summary.pivot(
            index='risk_tier',
            columns='benefit_group',
            values='pct_within_risk_tier',
        )
        .reindex(risk_tier_order)
        .fillna(0)
    )

    counts = (
        tier_totals.set_index('risk_tier')
        .reindex(risk_tier_order)['risk_tier_members']
        .fillna(0)
        .astype(int)
    )

    fig, ax = plt.subplots(figsize=(9.2, 6.0))

    bottom = np.zeros(len(pct))
    x = np.arange(len(pct.index))

    for group in benefit_group_order:
        values = pct[group].to_numpy() if group in pct else np.zeros(len(pct))

        ax.bar(
            x,
            values,
            bottom=bottom,
            label=group,
            color=benefit_group_colors[group],
            edgecolor='white',
            linewidth=0.8,
        )

        for idx, value in enumerate(values):
            if value >= 0.07:
                label_color = 'white' if group == 'High benefit' else '#222222'
                ax.text(
                    idx,
                    bottom[idx] + value / 2,
                    f'{value:.0%}',
                    ha='center',
                    va='center',
                    fontsize=9,
                    color=label_color,
                )

        bottom += values

    ax.set_xticks(x)

    ax.set_xticklabels(
        [f'{risk_tier_display[tier]}\n(n={counts.loc[tier]})' for tier in risk_tier_order]
    )

    ax.set_ylim(0, 1)
    ax.set_xlabel('Original Oregon risk tier (1 = lowest, 5 = highest)', labelpad=14)
    ax.set_ylabel('Percent of members within risk tier')
    ax.set_title(chart_title)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda y, _: f'{y:.0%}')
    )

    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.22),
        ncol=3,
        frameon=False,
    )

    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)

    fig.tight_layout(rect=[0, 0.08, 1, 1])

    chart_path = output_dir / f'dashboard_{file_prefix}_risk_tier_by_benefit_group.png'
    fig.savefig(chart_path, dpi=200, bbox_inches='tight')
    plt.close(fig)

    return summary


tlearner_xgboost_risk_tier_benefit_group_summary = build_risk_tier_benefit_group_outputs(
    results_test_xgboost,
    xgboost_output_folder,
    'tlearner',
    'XGBoost T-learner benefit group distribution by risk tier (test set)',
)

tlearner_risk_tier_benefit_group_summary = build_risk_tier_benefit_group_outputs(
    results_test_glmnet,
    glmnet_output_folder,
    'tlearner',
    'GLMNet T-learner benefit group distribution by risk tier (test set)',
)

xlearner_risk_tier_benefit_group_summary = build_risk_tier_benefit_group_outputs(
    results_test_xlearner_glmnet,
    xlearner_glmnet_output_folder,
    'xlearner',
    'GLMNet X-learner benefit group distribution by risk tier (test set)',
)

print('Oregon risk-tier definitions (original assignments; no current_risk_score derivation):')
display(risk_tier_thresholds)

print('Members excluded from the 1-5 risk-tier population summary because tier was missing or invalid:', unassigned_risk_tier_members)
print('Risk tier population summary (original Oregon tiers 1-5):')
display(risk_tier_population_summary)

print('T-learner risk tier by benefit group summary:')
display(tlearner_risk_tier_benefit_group_summary)

print('X-learner risk tier by benefit group summary:')
display(xlearner_risk_tier_benefit_group_summary)


---
## WRITE-UP OUTPUT FILE INDEX
---


In [ ]:
writeup_output_files = [
    output_folder / 'data_review_summary.csv',
    output_folder / 'preprocessing_audit_summary.csv',
    output_folder / 'preprocessing_column_audit.csv',
    output_folder / 'preprocessing_split_distribution.csv',
    output_folder / 'xgboost_observed_gap_by_decile_comparison.csv',
    predictor_distribution_folder / 'predictor_data_dictionary.csv',
    predictor_distribution_folder / 'numeric_predictor_summary.csv',
    predictor_distribution_folder / 'categorical_predictor_summary.csv',
    output_folder / 'Predictor_Distributions' / 'numeric_predictor_distributions.pdf',
    output_folder / 'Predictor_Distributions' / 'categorical_predictor_distributions.pdf',
    output_folder / 'Predictor_Distributions' / 'predictor_distribution_visual_index.csv',
    output_folder / 'model_evaluation_summary.csv',
    output_folder / 'model_recommendation_summary.csv',
    output_folder / 'factual_event_count_summary.csv',
    output_folder / 'factual_prediction_separation.csv',
    output_folder / 'factual_prediction_ranges.csv',
    output_folder / 'risk_tier_population_summary.csv',
    output_folder / 'risk_tier_thresholds.csv',
]

if 'xlearner_root_folder' in globals():
    writeup_output_files.append(xlearner_root_folder / 'xlearner_vs_tlearner_consistency_summary.csv')
    for folder in [xlearner_xgboost_output_folder, xlearner_glmnet_output_folder]:
        writeup_output_files.extend(
            [
                folder / 'xlearner_scored_test_output.csv',
                folder / 'xlearner_decile_summary.csv',
                folder / 'treatment_outcome_benefit_summary.csv',
            ]
        )


    writeup_output_files.extend(
        [
            xlearner_glmnet_output_folder / 'xlearner_risk_tier_benefit_group_summary.csv',
            xlearner_glmnet_output_folder / 'dashboard_xlearner_risk_tier_by_benefit_group.png',
        ]
    )
    # XGBoost X-Learner parity outputs
    writeup_output_files.extend(
        [
            xlearner_xgboost_output_folder / 'dashboard_avg_benefit_by_decile.png',
            xlearner_xgboost_output_folder / 'xlearner_roi_by_decile.csv',
            xlearner_xgboost_output_folder / 'xlearner_risk_tier_benefit_group_summary.csv',
            xlearner_xgboost_output_folder / 'cumulative_gross_savings_by_targeting.csv',
            xlearner_xgboost_output_folder / 'marginal_gross_savings_by_targeting.csv',
            xlearner_xgboost_output_folder / 'shap_importance_benefit_score.csv',
            xlearner_xgboost_output_folder / 'xlearner_benefit_driver_importance.csv',
            xlearner_xgboost_output_folder / 'dashboard_shap_benefit_score.png',
            xlearner_xgboost_output_folder / 'dashboard_xlearner_benefit_drivers.png',
            xlearner_xgboost_output_folder / 'dashboard_xlearner_risk_tier_by_benefit_group.png',
            xlearner_xgboost_output_folder / 'uplift_observed_gap_by_decile.csv',
            xlearner_xgboost_output_folder / 'dashboard_observed_gap_by_decile.png',
            xlearner_xgboost_output_folder / 'dashboard_cumulative_gross_savings_targeting.png',
            xlearner_xgboost_output_folder / 'dashboard_marginal_gross_savings_advantage_vs_current_risk.png',
        ]
    )
writeup_output_files.extend(
    [
        overlap_weighted_xgboost_output_folder / 'overlap_weighted_scored_test_output.csv',
        overlap_weighted_xgboost_output_folder / 'uplift_decile_summary.csv',
        overlap_weighted_xgboost_output_folder / 'treatment_outcome_benefit_summary.csv',
        overlap_weighted_xgboost_output_folder / 'dashboard_avg_benefit_by_decile.png',
        overlap_weighted_xgboost_output_folder / 'uplift_observed_gap_by_decile.csv',
        overlap_weighted_xgboost_output_folder / 'dashboard_observed_gap_by_decile.png',
    ]
)
for folder in [xgboost_output_folder, glmnet_output_folder]:
    writeup_output_files.extend(
        [
            folder / 'uplift_decile_summary.csv',
            folder / 'treatment_outcome_benefit_summary.csv',
            folder / 'uplift_roi_by_decile.csv',
            folder / 'model_brier_scores.csv',
            folder / 'calibration_summary.csv',
            folder / 'calibration_by_decile.csv',
            folder / 'uplift_observed_gap_by_decile.csv',
            folder / 'uplift_curve_by_decile.csv',
            folder / 'uplift_curve_summary.csv',
            folder / 'top_benefit_decile_summary.csv',
            folder / 'shap_importance_treated_control_models.csv',
            folder / 'shap_importance_benefit_score.csv',
            folder / 'dashboard_calibration_plot.png',
            folder / 'dashboard_observed_gap_by_decile.png',
            folder / 'dashboard_uplift_curve_by_decile.png',
            folder / 'dashboard_shap_benefit_score.png',
        ]
    )


writeup_output_files.extend(
    [
        glmnet_output_folder / 'tlearner_risk_tier_benefit_group_summary.csv',
        glmnet_output_folder / 'dashboard_tlearner_risk_tier_by_benefit_group.png',
    ]
)
writeup_output_index = pd.DataFrame(
    {
        'output_file': [str(path) for path in writeup_output_files],
        'exists': [path.exists() for path in writeup_output_files],
    }
)

print('Write-up output file index:')
display(writeup_output_index)
print('\n'.join(writeup_output_index['output_file']))



---
## XGBOOST X-LEARNER FEATURE PARITY
---

This section brings the XGBoost X-Learner output set to parity with the GLMNet X-Learner.
It generates all model-agnostic artifacts (decile dashboards, ROI, risk-tier segmentation,
TreeSHAP benefit-driver importance, cumulative/marginal targeting savings) using shared
reporting functions.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Shared X-Learner reporting functions (model-agnostic)
# ═══════════════════════════════════════════════════════════════════════════════


def generate_xlearner_decile_dashboard(decile_df, output_dir, model_label, benefit_y_max=None):
    """Generate standard decile dashboard charts for any X-Learner."""
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'avg_benefit_score',
        f'{model_label} X-Learner: Average Predicted Benefit by Uplift Decile',
        'Uplift Decile: 1 = Highest Predicted Benefit',
        'Average Benefit Score',
        output_dir / 'dashboard_avg_benefit_by_decile.png',
        y_max=benefit_y_max,
    )


def generate_xlearner_roi(decile_df, output_dir, model_label):
    """Generate ROI summary CSV and chart for any X-Learner."""
    roi_df = build_roi_summary(decile_df)
    roi_df.to_csv(output_dir / 'xlearner_roi_by_decile.csv', index=False)
    save_bar_chart(
        roi_df,
        'uplift_decile',
        'net_savings',
        f'{model_label} X-Learner: Estimated Net Savings by Uplift Decile',
        'Uplift Decile',
        'Estimated Net Savings',
        output_dir / 'dashboard_xlearner_roi_net_savings_by_decile.png',
    )
    print(f'{model_label} X-Learner ROI saved to:', output_dir)
    return roi_df


def generate_xlearner_risk_tier_outputs(results_df, output_dir, model_label):
    """Generate risk-tier by benefit-group summary for any X-Learner."""
    if 'risk_tier' not in results_df.columns or 'current_risk_score' not in results_df.columns:
        print(f'Skipped {model_label} X-Learner risk-tier outputs: required columns missing.')
        return None
    summary = build_risk_tier_benefit_group_outputs(
        results_df,
        output_dir,
        'xlearner',
        f'{model_label} X-Learner benefit group distribution by risk tier (test set)',
    )
    print(f'{model_label} X-Learner risk-tier outputs saved to:', output_dir)
    return summary


def generate_xlearner_cumulative_targeting(results_df, output_dir, model_label):
    """Generate cumulative and marginal gross savings targeting analysis."""
    n_total = len(results_df)
    if n_total == 0:
        return None, None, None
    decile_size = max(1, n_total // 10)

    ranking_specs = [
        ('Uplift score', 'benefit_score', False),
    ]
    if 'current_risk_score' in results_df.columns:
        ranking_specs.append(('Current risk score', 'current_risk_score', False))

    rows = []
    for approach, sort_col, ascending in ranking_specs:
        ranked = results_df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
        for decile in range(1, 11):
            top_n = n_total if decile == 10 else decile * decile_size
            selected = ranked.head(top_n)
            avoided = float(selected['benefit_score'].sum())
            rows.append({
                'targeting_approach': approach,
                'through_decile': decile,
                'population_fraction_targeted': top_n / n_total,
                'n': top_n,
                'cumulative_estimated_ed_visits_avoided': avoided,
                'cumulative_gross_savings': avoided * cost_per_ed_visit,
            })

    cumulative_df = pd.DataFrame(rows)
    cumulative_df.to_csv(output_dir / 'cumulative_gross_savings_by_targeting.csv', index=False)

    # Summary at top 50%
    summary_rows = []
    for approach in cumulative_df['targeting_approach'].unique():
        subset = cumulative_df[cumulative_df['targeting_approach'] == approach]
        top50 = subset[subset['population_fraction_targeted'] <= 0.51].iloc[-1]
        summary_rows.append({
            'targeting_approach': approach,
            'population_fraction_targeted': top50['population_fraction_targeted'],
            'n': int(top50['n']),
            'cumulative_gross_savings': top50['cumulative_gross_savings'],
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(output_dir / 'cumulative_gross_savings_summary_top50.csv', index=False)

    # Marginal savings by targeting approach
    marginal_rows = []
    for approach in cumulative_df['targeting_approach'].unique():
        subset = cumulative_df[cumulative_df['targeting_approach'] == approach].sort_values('through_decile')
        prev = 0.0
        for _, row in subset.iterrows():
            marginal = row['cumulative_gross_savings'] - prev
            marginal_rows.append({
                'targeting_approach': approach,
                'decile': int(row['through_decile']),
                'marginal_gross_savings': marginal,
                'cumulative_gross_savings': row['cumulative_gross_savings'],
            })
            prev = row['cumulative_gross_savings']
    marginal_df = pd.DataFrame(marginal_rows)
    marginal_df.to_csv(output_dir / 'marginal_gross_savings_by_targeting.csv', index=False)

    # Marginal advantage vs current risk
    if 'Current risk score' in cumulative_df['targeting_approach'].values:
        uplift_marginals = marginal_df[marginal_df['targeting_approach'] == 'Uplift score'].set_index('decile')['marginal_gross_savings']
        risk_marginals = marginal_df[marginal_df['targeting_approach'] == 'Current risk score'].set_index('decile')['marginal_gross_savings']
        advantage_rows = []
        for decile in range(1, 11):
            u = uplift_marginals.get(decile, 0)
            r = risk_marginals.get(decile, 0)
            advantage_rows.append({
                'decile': decile,
                'uplift_marginal_gross_savings': u,
                'risk_marginal_gross_savings': r,
                'marginal_advantage': u - r,
            })
        advantage_df = pd.DataFrame(advantage_rows)
        advantage_df.to_csv(output_dir / 'marginal_gross_savings_advantage_vs_current_risk.csv', index=False)
    else:
        advantage_df = None

    # Cumulative savings chart
    chart_df = cumulative_df[cumulative_df['population_fraction_targeted'] <= 0.51]
    fig, ax = plt.subplots(figsize=(8.5, 5.25))
    for approach in chart_df['targeting_approach'].unique():
        approach_df = chart_df[chart_df['targeting_approach'] == approach]
        ax.plot(
            approach_df['population_fraction_targeted'],
            approach_df['cumulative_gross_savings'],
            marker='o',
            label=approach,
        )
    ax.set_title(f'{model_label} X-Learner: Cumulative Gross Savings by Targeting Approach')
    ax.set_xlabel('Population Fraction Targeted')
    ax.set_ylabel('Cumulative Gross Savings ($)')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_dir / 'dashboard_cumulative_gross_savings_targeting.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

    # Marginal advantage chart
    if advantage_df is not None:
        fig, ax = plt.subplots(figsize=(8.5, 5.25))
        ax.bar(
            advantage_df['decile'].astype(str),
            advantage_df['marginal_advantage'],
            color='#4c78a8',
        )
        ax.axhline(0, color='gray', linewidth=1)
        ax.set_title(f'{model_label} X-Learner: Marginal Gross Savings Advantage vs Current Risk')
        ax.set_xlabel('Decile')
        ax.set_ylabel('Marginal Gross Savings Advantage ($)')
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        fig.savefig(output_dir / 'dashboard_marginal_gross_savings_advantage_vs_current_risk.png', dpi=200, bbox_inches='tight')
        plt.close(fig)

    print(f'{model_label} X-Learner targeting/savings outputs saved to:', output_dir)
    return cumulative_df, marginal_df, advantage_df


def generate_xlearner_benefit_shap_xgboost(tau_treated_model, tau_control_model, propensity_scores, x_matrix, output_dir, model_label):
    """Generate TreeSHAP benefit-driver outputs for XGBoost X-Learner effect models."""
    # Get TreeSHAP contributions from both effect models
    dmatrix = make_dmatrix(x_matrix)
    tau_treated_contribs = tau_treated_model.predict(dmatrix, pred_contribs=True)
    tau_control_contribs = tau_control_model.predict(dmatrix, pred_contribs=True)

    shap_columns = [*x_matrix.columns, 'BIAS']
    tau_treated_shap = pd.DataFrame(tau_treated_contribs, columns=shap_columns, index=x_matrix.index)
    tau_control_shap = pd.DataFrame(tau_control_contribs, columns=shap_columns, index=x_matrix.index)

    # Weighted combination: benefit = (1 - e(x)) * tau1(x) + e(x) * tau0(x)
    treated_weight = 1 - propensity_scores
    control_weight = propensity_scores

    weighted_treated_shap = tau_treated_shap.mul(treated_weight, axis=0)
    weighted_control_shap = tau_control_shap.mul(control_weight, axis=0)
    benefit_shap = weighted_treated_shap + weighted_control_shap
    benefit_shap_no_bias = benefit_shap.drop(columns=['BIAS'], errors='ignore')

    # Importance summary
    importance_df = pd.DataFrame({
        'feature': benefit_shap_no_bias.columns,
        'mean_abs_benefit_shap': benefit_shap_no_bias.abs().mean(axis=0).to_numpy(),
        'mean_signed_benefit_shap': benefit_shap_no_bias.mean(axis=0).to_numpy(),
        'pct_positive_benefit_shap': benefit_shap_no_bias.gt(0).mean(axis=0).to_numpy(),
        'importance_type': 'xgboost_xlearner_weighted_treeshap',
    }).sort_values('mean_abs_benefit_shap', ascending=False).reset_index(drop=True)

    # Save outputs
    importance_df.to_csv(output_dir / 'shap_importance_benefit_score.csv', index=False)
    importance_df.to_csv(output_dir / 'xlearner_benefit_driver_importance.csv', index=False)
    benefit_shap_no_bias.to_csv(output_dir / 'xgboost_xlearner_member_benefit_shap_values.csv', index=True, index_label='row_index')

    # Reconciliation
    baseline_treated = float(tau_treated_shap['BIAS'].mean()) if 'BIAS' in tau_treated_shap.columns else 0.0
    baseline_control = float(tau_control_shap['BIAS'].mean()) if 'BIAS' in tau_control_shap.columns else 0.0
    avg_treated_weight = float(np.mean(treated_weight))
    avg_control_weight = float(np.mean(control_weight))
    weighted_baseline = baseline_treated * avg_treated_weight + baseline_control * avg_control_weight
    signed_sum = float(importance_df['mean_signed_benefit_shap'].sum())

    reconciliation_df = pd.DataFrame([
        {'metric': 'weighted_shap_baseline', 'value': weighted_baseline},
        {'metric': 'sum_mean_signed_feature_shap', 'value': signed_sum},
        {'metric': 'baseline_plus_signed_shap_sum', 'value': weighted_baseline + signed_sum},
    ])
    reconciliation_df.to_csv(output_dir / 'xgboost_xlearner_global_benefit_shap_reconciliation.csv', index=False)

    # Global benefit SHAP importance chart
    top = importance_df.head(20).sort_values('mean_abs_benefit_shap')
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top['mean_abs_benefit_shap'])
    ax.set_title(f'{model_label} X-Learner: Top Drivers of Predicted Treatment Benefit (TreeSHAP)')
    ax.set_xlabel('Mean Absolute SHAP Contribution to Benefit')
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(output_dir / 'dashboard_shap_benefit_score.png', dpi=150)
    fig.savefig(output_dir / 'dashboard_xlearner_benefit_drivers.png', dpi=150)
    plt.close(fig)

    # Global benefit SHAP chart (matching GLMNet naming pattern)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top['mean_abs_benefit_shap'])
    ax.set_title(f'{model_label} X-Learner: Global SHAP Drivers of Benefit Score')
    ax.set_xlabel('Mean absolute SHAP contribution to benefit score')
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(output_dir / 'dashboard_xgboost_xlearner_global_benefit_shap.png', dpi=150)
    plt.close(fig)

    print(f'{model_label} X-Learner TreeSHAP benefit-driver outputs saved to:', output_dir)
    return importance_df, benefit_shap_no_bias, reconciliation_df


# ═══════════════════════════════════════════════════════════════════════════════
# Generate XGBoost X-Learner outputs using shared functions
# ═══════════════════════════════════════════════════════════════════════════════

# 1. Decile dashboard chart
generate_xlearner_decile_dashboard(
    decile_summary_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

# 2. ROI by decile
xgboost_xlearner_roi = generate_xlearner_roi(
    decile_summary_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

# 3. Risk tier by benefit group
xgboost_xlearner_risk_tier_summary = generate_xlearner_risk_tier_outputs(
    results_test_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

# 4. Cumulative and marginal targeting savings
(
    xgboost_xlearner_cumulative,
    xgboost_xlearner_marginal,
    xgboost_xlearner_advantage,
) = generate_xlearner_cumulative_targeting(
    results_test_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

# 5. TreeSHAP benefit-driver importance
(
    xgboost_xlearner_shap_importance,
    xgboost_xlearner_member_shap,
    xgboost_xlearner_shap_reconciliation,
) = generate_xlearner_benefit_shap_xgboost(
    xgb_tau_treated_model,
    xgb_tau_control_model,
    propensity_test,
    x_test,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

print()
print('XGBoost X-Learner feature parity outputs complete.')
print('Files in XGBoost X-Learner folder:')
print('\\n'.join(str(p) for p in sorted(xlearner_xgboost_output_folder.iterdir())))


---
## REGENERATE GLMNET X-LEARNER OUTPUTS VIA SHARED FUNCTIONS
---

Use the same shared reporting functions to regenerate the GLMNet X-Learner outputs,
confirming functional equivalence and consolidating duplicate logic.


In [ ]:
# Regenerate GLMNet X-Learner outputs through the same shared functions.
# The GLMNet X-Learner already has these files, but regenerating confirms parity.

if results_test_xlearner_glmnet is not None and decile_summary_xlearner_glmnet is not None:
    # Decile dashboard
    generate_xlearner_decile_dashboard(
        decile_summary_xlearner_glmnet,
        xlearner_glmnet_output_folder,
        'GLMNet',
    )

    # Cumulative/marginal savings (regenerate using shared function)
    generate_xlearner_cumulative_targeting(
        results_test_xlearner_glmnet,
        xlearner_glmnet_output_folder,
        'GLMNet',
    )

    print()
    print('GLMNet X-Learner outputs regenerated via shared functions.')
else:
    print('Skipped GLMNet X-Learner regeneration: models not available.')


In [ ]:
# Build an explicit PRP-to-Oregon output parity manifest without modifying PRP outputs.
prp_python_root = PROJECT_ROOT / 'Outputs' / 'Uplift' / 'Python'
incidental_parts = {'.ipynb_checkpoints', '__pycache__'}
synthetic_tokens = ('true_benefit', 'true_driver')
prp_files = [
    path for path in prp_python_root.rglob('*')
    if path.is_file() and not incidental_parts.intersection(path.parts) and path.name != '.gitkeep'
]
parity_rows = []
expected_funds_paths = set()
for prp_path in sorted(prp_files):
    relative = prp_path.relative_to(prp_python_root)
    funds_path = output_folder / relative
    relative_text = relative.as_posix()
    applicability = 'applicable'
    explanation = ''
    if any(token in relative_text.lower() for token in synthetic_tokens):
        applicability = 'not_applicable'
        explanation = 'Synthetic counterfactual ground truth is unavailable for Oregon 2-Year.'
    elif relative.parts[:2] in [('Predictor_Distributions', 'Categorical'), ('Predictor_Distributions', 'Numeric')] and not funds_path.exists():
        applicability = 'not_applicable'
        explanation = 'Predictor-specific plot is schema-dependent and this PRP predictor is not retained for Oregon.'
    if applicability == 'applicable':
        expected_funds_paths.add(funds_path.resolve())
    parity_rows.append({
        'corresponding_prp_relative_path': relative_text,
        'expected_funds_relative_path': relative_text,
        'artifact_category': relative.parts[0] if len(relative.parts) > 1 else 'Python root',
        'applicability_status': applicability,
        'generated_status': funds_path.exists() if applicability == 'applicable' else False,
        'explanation': explanation,
    })

existing_funds_files = [
    path for path in output_folder.rglob('*')
    if path.is_file() and path.name != 'output_parity_manifest.csv' and not incidental_parts.intersection(path.parts)
]
for funds_path in sorted(existing_funds_files):
    if funds_path.resolve() not in expected_funds_paths:
        relative = funds_path.relative_to(output_folder).as_posix()
        parity_rows.append({
            'corresponding_prp_relative_path': '',
            'expected_funds_relative_path': relative,
            'artifact_category': 'Oregon-specific addition',
            'applicability_status': 'oregon_specific',
            'generated_status': True,
            'explanation': 'Oregon-specific preprocessing or schema artifact.',
        })

output_parity_manifest = pd.DataFrame(parity_rows)
output_parity_manifest.to_csv(output_folder / 'output_parity_manifest.csv', index=False)
missing_applicable = output_parity_manifest.query("applicability_status == 'applicable' and generated_status == False")
display(output_parity_manifest.groupby(['applicability_status', 'generated_status']).size().rename('n').reset_index())
if not missing_applicable.empty:
    print('Applicable PRP counterparts not generated:')
    display(missing_applicable)
else:
    print('All applicable PRP output counterparts were generated.')

final_encoded_leaks = [
    column for column in combined_matrix.columns
    if any(canonical_column_name(column).startswith(item) for item in forbidden_feature_canonical)
]
assert not final_encoded_leaks, f'Final leakage audit failed: {final_encoded_leaks}'
print('Final encoded-feature leakage audit passed.')
